In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier, ExtraTreesClassifier
from sklearn.neighbors import KNeighborsClassifier, NearestCentroid
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import BernoulliNB
from sklearn.svm import SVC
import xgboost as xgb
import lightgbm as exc
from sklearn import metrics, clone
import h5py
import joblib
from sklearn.metrics import *
import math
import optuna
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedKFold
import sklearn.metrics

from pathlib import Path
from warnings import filterwarnings
filterwarnings('ignore')

In [2]:
df_internal = pd.read_pickle('FingerprintsAll.csv')
print(len(df_internal))
df_internal.head()

6403


,Name,fp_MACCS,fp_Morgan_512B,fp_Morgan_512B_r3,fp_Morgan_1024B,fp_Morgan_1024B_r3,fp_MAP4_512B,fp_MAP4_512B_r3,fp_MAP4_1024B,fp_MAP4_1024B_r3,Toxicity
0,DB00014,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, ...","[0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",0
1,DB00035,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ...","[0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, ...","[0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, ...","[0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, ...","[0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",0
2,DB00050,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, ...","[0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",0
3,DB00091,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, ...","[0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",0
4,DB00093,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ...","[0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, ...","[0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, ...","[0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, ...","[0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",0


## AMES Set

In [3]:
df = pd.read_pickle('tdc_ames_FP.csv')
print(len(df))
df.head()

6444


,Name,SMILES,fp_MACCS,Toxicity
0,AMES_0,O=[N+]([O-])c1ccc2ccc3ccc([N+](=O)[O-])c4c5ccc...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1.0
1,AMES_1,O=[N+]([O-])c1c2c(c3ccc4cccc5ccc1c3c45)CCCC2,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1.0
2,AMES_2,O=c1c2ccccc2c(=O)c2c1ccc1c2[nH]c2c3c(=O)c4cccc...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0.0
3,AMES_3,[N-]=[N+]=CC(=O)NCC(=O)NN,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1.0
4,AMES_4,[N-]=[N+]=C1C=NC(=O)NC1=O,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1.0


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6444 entries, 0 to 6443
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Name      6444 non-null   object 
 1   SMILES    6444 non-null   object 
 2   fp_MACCS  6444 non-null   object 
 3   Toxicity  6444 non-null   float64
dtypes: float64(1), object(3)
memory usage: 201.5+ KB


In [5]:
print("Number of toxic molecules in external set:", len(df[df["Toxicity"] == 1.0]))
print("Number of nontoxic molecules in external set:", len(df[df["Toxicity"] == 0.0]))

Number of toxic molecules in external set: 3654
Number of nontoxic molecules in external set: 2790


#### Test Each Model One by One

In [6]:
#By using Morgan fingerprints with radius of 3 and 1025 bits
X_internal = np.array(list((df_internal['fp_MACCS']))).astype(float)
print(X_internal.shape)
y_internal=df_internal.Toxicity
print(y_internal.head())

(6403, 167)
0    0
1    0
2    0
3    0
4    0
Name: Toxicity, dtype: int64


In [7]:
#By using Morgan fingerprints with radius of 3 and 1025 bits
X = np.array(list((df['fp_MACCS']))).astype(float)
print(X.shape)
y_real=df.Toxicity
print(y_real.head())

(6444, 167)
0    1.0
1    1.0
2    0.0
3    1.0
4    1.0
Name: Toxicity, dtype: float64


In [8]:
optimizedCV_clf_rf =  joblib.load("FinalModels/RF_optimized_full.joblib")
y_pred_rf = optimizedCV_clf_rf.predict(X)

#add to a dataframe
df_preds = df[["Name", "SMILES", "Toxicity"]]
df_preds["pred_rf"] = y_pred_rf
df_preds["pred_rf_proba"] = optimizedCV_clf_rf.predict_proba(X).tolist()
df_preds["pred_rf_proba"] = df_preds["pred_rf_proba"].str[1]

conf_matrix_rf = confusion_matrix(y_real, y_pred_rf) 
TP_rf = conf_matrix_rf[1][1]
TN_rf = conf_matrix_rf[0][0]
FP_rf = conf_matrix_rf[0][1] 
FN_rf = conf_matrix_rf[1][0]
Accuracy_rf = accuracy_score(y_real, y_pred_rf)
Precision_rf = precision_score(y_real, y_pred_rf)
Sensitivity_rf = recall_score(y_real, y_pred_rf)
Specificity_rf = round( TN_rf / (TN_rf+FP_rf),4 )
f1_scores_rf = f1_score(y_real, y_pred_rf)
f1_scores_W_rf = f1_score(y_real, y_pred_rf, average="weighted")
f1_scores_M_rf = f1_score(y_real, y_pred_rf, average="macro")
BA_scores_rf = balanced_accuracy_score(y_real, y_pred_rf)
MCC_rf= matthews_corrcoef(y_real, y_pred_rf)
NPV_rf= round( TN_rf / (TN_rf+FN_rf),4 )
ROC_AUC_rf = roc_auc_score(y_real, y_pred_rf)

mat_met = pd.DataFrame({'Metric':['TP','TN','FP','FN','Accuracy','Precision','Sensitivity','Specificity','F1 score','F1 score (weighted)',
                            'F1 score (macro)', 'Balanced Accuracy','MCC','NPV','ROC_AUC'],     
                        'rf':[np.mean(TP_rf), np.mean(TN_rf),
                                    np.mean(FP_rf), np.mean(FN_rf),
                                    np.mean(Accuracy_rf),
                                    np.mean(Precision_rf),
                                    np.mean(Sensitivity_rf),
                                    np.mean(Specificity_rf),
                                    np.mean(f1_scores_rf),
                                    np.mean(f1_scores_W_rf), 
                                    np.mean(f1_scores_M_rf), 
                                    np.mean(BA_scores_rf), 
                                    np.mean(MCC_rf),
                                    np.mean(NPV_rf),
                                    np.mean(ROC_AUC_rf)],
                   }) 
    
print(mat_met)

                 Metric           rf
0                    TP  2710.000000
1                    TN   875.000000
2                    FP  1915.000000
3                    FN   944.000000
4              Accuracy     0.556331
5             Precision     0.585946
6           Sensitivity     0.741653
7           Specificity     0.313600
8              F1 score     0.654668
9   F1 score (weighted)     0.535614
10     F1 score (macro)     0.517180
11    Balanced Accuracy     0.527637
12                  MCC     0.060845
13                  NPV     0.481000
14              ROC_AUC     0.527637


In [9]:
optimizedCV_clf_lgbm =  joblib.load("FinalModels/LGBM_optimized_full.joblib")
y_pred_lgbm = optimizedCV_clf_lgbm.predict(X)
df_preds["pred_lgbm"] = y_pred_lgbm
df_preds["pred_lgbm_proba"] = optimizedCV_clf_lgbm.predict_proba(X).tolist()
df_preds["pred_lgbm_proba"] = df_preds["pred_lgbm_proba"].str[1]

conf_matrix_lgbm = confusion_matrix(y_real, y_pred_lgbm) 
TP_lgbm = conf_matrix_lgbm[1][1]
TN_lgbm = conf_matrix_lgbm[0][0]
FP_lgbm = conf_matrix_lgbm[0][1] 
FN_lgbm = conf_matrix_lgbm[1][0]
Accuracy_lgbm = accuracy_score(y_real, y_pred_lgbm)
Precision_lgbm = precision_score(y_real, y_pred_lgbm)
Sensitivity_lgbm = recall_score(y_real, y_pred_lgbm)
Specificity_lgbm = round( TN_lgbm / (TN_lgbm+FP_lgbm),4 )
f1_scores_lgbm = f1_score(y_real, y_pred_lgbm)
f1_scores_W_lgbm = f1_score(y_real, y_pred_lgbm, average="weighted")
f1_scores_M_lgbm = f1_score(y_real, y_pred_lgbm, average="macro")
BA_scores_lgbm = balanced_accuracy_score(y_real, y_pred_lgbm)
MCC_lgbm= matthews_corrcoef(y_real, y_pred_lgbm)
NPV_lgbm= round( TN_lgbm / (TN_lgbm+FN_lgbm),4 )
ROC_AUC_lgbm = roc_auc_score(y_real, y_pred_lgbm)

lgbm = pd.DataFrame({  'lgbm':[np.mean(TP_lgbm), np.mean(TN_lgbm),
                                    np.mean(FP_lgbm), np.mean(FN_lgbm),
                                    np.mean(Accuracy_lgbm),
                                    np.mean(Precision_lgbm),
                                    np.mean(Sensitivity_lgbm),
                                    np.mean(Specificity_lgbm),
                                    np.mean(f1_scores_lgbm),
                                    np.mean(f1_scores_W_lgbm), 
                                    np.mean(f1_scores_M_lgbm), 
                                    np.mean(BA_scores_lgbm), 
                                    np.mean(MCC_lgbm),
                                    np.mean(NPV_lgbm),
                                    np.mean(ROC_AUC_lgbm)],
                 }) 

mat_met['lgbm'] = lgbm  
    
print(mat_met)

                 Metric           rf         lgbm
0                    TP  2710.000000  2859.000000
1                    TN   875.000000   833.000000
2                    FP  1915.000000  1957.000000
3                    FN   944.000000   795.000000
4              Accuracy     0.556331     0.572936
5             Precision     0.585946     0.593646
6           Sensitivity     0.741653     0.782430
7           Specificity     0.313600     0.298600
8              F1 score     0.654668     0.675089
9   F1 score (weighted)     0.535614     0.546068
10     F1 score (macro)     0.517180     0.526091
11    Balanced Accuracy     0.527637     0.540498
12                  MCC     0.060845     0.092360
13                  NPV     0.481000     0.511700
14              ROC_AUC     0.527637     0.540498


In [10]:
optimizedCV_clf_svm =  joblib.load("FinalModels/SVC_optimized_full.joblib")
y_pred_svm = optimizedCV_clf_svm.predict(X)
df_preds["pred_svm"] = y_pred_svm
df_preds["pred_svm_proba"] = optimizedCV_clf_svm.predict_proba(X).tolist()
df_preds["pred_svm_proba"] = df_preds["pred_svm_proba"].str[1]

conf_matrix_svm = confusion_matrix(y_real, y_pred_svm) 
TP_svm = conf_matrix_svm[1][1]
TN_svm = conf_matrix_svm[0][0]
FP_svm = conf_matrix_svm[0][1] 
FN_svm = conf_matrix_svm[1][0]
Accuracy_svm = accuracy_score(y_real, y_pred_svm)
Precision_svm = precision_score(y_real, y_pred_svm)
Sensitivity_svm = recall_score(y_real, y_pred_svm)
Specificity_svm = round( TN_svm / (TN_svm+FP_svm),4 )
f1_scores_svm = f1_score(y_real, y_pred_svm)
f1_scores_W_svm = f1_score(y_real, y_pred_svm, average="weighted")
f1_scores_M_svm = f1_score(y_real, y_pred_svm, average="macro")
BA_scores_svm = balanced_accuracy_score(y_real, y_pred_svm)
MCC_svm= matthews_corrcoef(y_real, y_pred_svm)
NPV_svm= round( TN_svm / (TN_svm+FN_svm),4 )
ROC_AUC_svm = roc_auc_score(y_real, y_pred_svm)

svm = pd.DataFrame({  'svm':[np.mean(TP_svm), np.mean(TN_svm),
                                    np.mean(FP_svm), np.mean(FN_svm),
                                    np.mean(Accuracy_svm),
                                    np.mean(Precision_svm),
                                    np.mean(Sensitivity_svm),
                                    np.mean(Specificity_svm),
                                    np.mean(f1_scores_svm),
                                    np.mean(f1_scores_W_svm), 
                                    np.mean(f1_scores_M_svm), 
                                    np.mean(BA_scores_svm), 
                                    np.mean(MCC_svm),
                                    np.mean(NPV_svm),
                                    np.mean(ROC_AUC_svm)],
                 }) 

mat_met['svm'] = svm  
print(mat_met)

                 Metric           rf         lgbm          svm
0                    TP  2710.000000  2859.000000  2910.000000
1                    TN   875.000000   833.000000   820.000000
2                    FP  1915.000000  1957.000000  1970.000000
3                    FN   944.000000   795.000000   744.000000
4              Accuracy     0.556331     0.572936     0.578833
5             Precision     0.585946     0.593646     0.596311
6           Sensitivity     0.741653     0.782430     0.796388
7           Specificity     0.313600     0.298600     0.293900
8              F1 score     0.654668     0.675089     0.681978
9   F1 score (weighted)     0.535614     0.546068     0.549789
10     F1 score (macro)     0.517180     0.526091     0.529322
11    Balanced Accuracy     0.527637     0.540498     0.545147
12                  MCC     0.060845     0.092360     0.104356
13                  NPV     0.481000     0.511700     0.524300
14              ROC_AUC     0.527637     0.540498     0

In [11]:
optimizedCV_clf_ridge =  joblib.load("FinalModels/Ridge_optimized_full.joblib")

from sklearn.calibration import CalibratedClassifierCV
ridge_cal = CalibratedClassifierCV(
    optimizedCV_clf_ridge,
    method="sigmoid",   # Platt scaling
    cv="prefit"
)

ridge_cal.fit(X_internal, y_internal)

y_pred_ridge = ridge_cal.predict(X)
df_preds["pred_ridge"] = y_pred_ridge
df_preds["pred_ridge_proba"] = ridge_cal.predict_proba(X).tolist()
df_preds["pred_ridge_proba"] = df_preds["pred_ridge_proba"].str[1]
df_preds["pred_ridge"] = y_pred_ridge
#df_preds["pred_ridge_proba"] = optimizedCV_clf_ridge.decision_function(X).tolist()
#df_preds["pred_ridge_proba"] = df_preds["pred_ridge_proba"].str[1]

conf_matrix_ridge = confusion_matrix(y_real, y_pred_ridge) 
TP_ridge = conf_matrix_ridge[1][1]
TN_ridge = conf_matrix_ridge[0][0]
FP_ridge = conf_matrix_ridge[0][1] 
FN_ridge = conf_matrix_ridge[1][0]
Accuracy_ridge = accuracy_score(y_real, y_pred_ridge)
Precision_ridge = precision_score(y_real, y_pred_ridge)
Sensitivity_ridge = recall_score(y_real, y_pred_ridge)
Specificity_ridge = round( TN_ridge / (TN_ridge+FP_ridge),4 )
f1_scores_ridge = f1_score(y_real, y_pred_ridge)
f1_scores_W_ridge = f1_score(y_real, y_pred_ridge, average="weighted")
f1_scores_M_ridge = f1_score(y_real, y_pred_ridge, average="macro")
BA_scores_ridge = balanced_accuracy_score(y_real, y_pred_ridge)
MCC_ridge= matthews_corrcoef(y_real, y_pred_ridge)
NPV_ridge= round( TN_ridge / (TN_ridge+FN_ridge),4 )
ROC_AUC_ridge = ROC_AUC_ridge = roc_auc_score(y_real,
    optimizedCV_clf_ridge.decision_function(X))

ridge = pd.DataFrame({  'ridge':[np.mean(TP_ridge), np.mean(TN_ridge),
                                    np.mean(FP_ridge), np.mean(FN_ridge),
                                    np.mean(Accuracy_ridge),
                                    np.mean(Precision_ridge),
                                    np.mean(Sensitivity_ridge),
                                    np.mean(Specificity_ridge),
                                    np.mean(f1_scores_ridge),
                                    np.mean(f1_scores_W_ridge), 
                                    np.mean(f1_scores_M_ridge), 
                                    np.mean(BA_scores_ridge), 
                                    np.mean(MCC_ridge),
                                    np.mean(NPV_ridge),
                                    np.mean(ROC_AUC_ridge)],
                 }) 

mat_met['ridge'] = ridge  
print(mat_met)

                 Metric           rf         lgbm          svm        ridge
0                    TP  2710.000000  2859.000000  2910.000000  2920.000000
1                    TN   875.000000   833.000000   820.000000   823.000000
2                    FP  1915.000000  1957.000000  1970.000000  1967.000000
3                    FN   944.000000   795.000000   744.000000   734.000000
4              Accuracy     0.556331     0.572936     0.578833     0.580850
5             Precision     0.585946     0.593646     0.596311     0.597504
6           Sensitivity     0.741653     0.782430     0.796388     0.799124
7           Specificity     0.313600     0.298600     0.293900     0.295000
8              F1 score     0.654668     0.675089     0.681978     0.683761
9   F1 score (weighted)     0.535614     0.546068     0.549789     0.551661
10     F1 score (macro)     0.517180     0.526091     0.529322     0.531206
11    Balanced Accuracy     0.527637     0.540498     0.545147     0.547053
12          

In [12]:
optimizedCV_clf_knn =  joblib.load("FinalModels/KNN_optimized_full.joblib")
y_pred_knn = optimizedCV_clf_knn.predict(X)
df_preds["pred_knn"] = y_pred_knn
df_preds["pred_knn_proba"] = optimizedCV_clf_knn.predict_proba(X).tolist()
df_preds["pred_knn_proba"] = df_preds["pred_knn_proba"].str[1]

conf_matrix_knn = confusion_matrix(y_real, y_pred_knn) 
TP_knn = conf_matrix_knn[1][1]
TN_knn = conf_matrix_knn[0][0]
FP_knn = conf_matrix_knn[0][1] 
FN_knn = conf_matrix_knn[1][0]
Accuracy_knn = accuracy_score(y_real, y_pred_knn)
Precision_knn = precision_score(y_real, y_pred_knn)
Sensitivity_knn = recall_score(y_real, y_pred_knn)
Specificity_knn = round( TN_knn / (TN_knn+FP_knn),4 )
f1_scores_knn = f1_score(y_real, y_pred_knn)
f1_scores_W_knn = f1_score(y_real, y_pred_knn, average="weighted")
f1_scores_M_knn = f1_score(y_real, y_pred_knn, average="macro")
BA_scores_knn = balanced_accuracy_score(y_real, y_pred_knn)
MCC_knn= matthews_corrcoef(y_real, y_pred_knn)
NPV_knn= round( TN_knn / (TN_knn+FN_knn),4 )
ROC_AUC_knn = roc_auc_score(y_real, y_pred_knn)

knn = pd.DataFrame({  'knn':[np.mean(TP_knn), np.mean(TN_knn),
                                    np.mean(FP_knn), np.mean(FN_knn),
                                    np.mean(Accuracy_knn),
                                    np.mean(Precision_knn),
                                    np.mean(Sensitivity_knn),
                                    np.mean(Specificity_knn),
                                    np.mean(f1_scores_knn),
                                    np.mean(f1_scores_W_knn), 
                                    np.mean(f1_scores_M_knn), 
                                    np.mean(BA_scores_knn), 
                                    np.mean(MCC_knn),
                                    np.mean(NPV_knn),
                                    np.mean(ROC_AUC_knn)],
                 }) 

mat_met['knn'] = knn  
print(mat_met)

                 Metric           rf         lgbm          svm        ridge  \
0                    TP  2710.000000  2859.000000  2910.000000  2920.000000   
1                    TN   875.000000   833.000000   820.000000   823.000000   
2                    FP  1915.000000  1957.000000  1970.000000  1967.000000   
3                    FN   944.000000   795.000000   744.000000   734.000000   
4              Accuracy     0.556331     0.572936     0.578833     0.580850   
5             Precision     0.585946     0.593646     0.596311     0.597504   
6           Sensitivity     0.741653     0.782430     0.796388     0.799124   
7           Specificity     0.313600     0.298600     0.293900     0.295000   
8              F1 score     0.654668     0.675089     0.681978     0.683761   
9   F1 score (weighted)     0.535614     0.546068     0.549789     0.551661   
10     F1 score (macro)     0.517180     0.526091     0.529322     0.531206   
11    Balanced Accuracy     0.527637     0.540498   

In [13]:
df_preds

,Name,SMILES,Toxicity,pred_rf,pred_rf_proba,pred_lgbm,pred_lgbm_proba,pred_svm,pred_svm_proba,pred_ridge,pred_ridge_proba,pred_knn,pred_knn_proba
0,AMES_0,O=[N+]([O-])c1ccc2ccc3ccc([N+](=O)[O-])c4c5ccc...,1.0,1,0.680702,1,0.976062,1,0.836607,1,0.899359,1,1.000000
1,AMES_1,O=[N+]([O-])c1c2c(c3ccc4cccc5ccc1c3c45)CCCC2,1.0,1,0.645614,1,0.946290,1,0.687504,1,0.873967,1,1.000000
2,AMES_2,O=c1c2ccccc2c(=O)c2c1ccc1c2[nH]c2c3c(=O)c4cccc...,0.0,1,0.768421,1,0.723822,1,0.819435,1,0.501393,1,0.714286
3,AMES_3,[N-]=[N+]=CC(=O)NCC(=O)NN,1.0,1,0.709649,1,0.887996,1,0.775291,1,0.806515,1,0.857143
4,AMES_4,[N-]=[N+]=C1C=NC(=O)NC1=O,1.0,1,0.698538,1,0.867479,1,0.814319,1,0.908299,1,0.714286
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6439,AMES_6439,CC(C)=C1CCC(C)CC1=O,0.0,1,0.774935,1,0.735004,1,0.659195,1,0.830495,1,0.857143
6440,AMES_6440,CC(CCc1ccccc1)c1ccccc1,0.0,1,0.822807,1,0.754054,1,0.646190,1,0.575176,1,1.000000
6441,AMES_6441,CCOP(=S)(CC)Sc1ccccc1,0.0,1,0.968421,1,0.993009,1,0.814835,1,0.916946,1,1.000000
6442,AMES_6442,C=C(C)C1CC=C(C)C(OC(C)=O)C1,0.0,1,0.796959,1,0.768768,1,0.700767,1,0.890627,1,0.857143


In [14]:
df_preds["mode"] = df_preds[["pred_rf", "pred_lgbm", "pred_svm","pred_ridge", "pred_knn" ]].mode(axis=1)
df_preds

,Name,SMILES,Toxicity,pred_rf,pred_rf_proba,pred_lgbm,pred_lgbm_proba,pred_svm,pred_svm_proba,pred_ridge,pred_ridge_proba,pred_knn,pred_knn_proba,mode
0,AMES_0,O=[N+]([O-])c1ccc2ccc3ccc([N+](=O)[O-])c4c5ccc...,1.0,1,0.680702,1,0.976062,1,0.836607,1,0.899359,1,1.000000,1
1,AMES_1,O=[N+]([O-])c1c2c(c3ccc4cccc5ccc1c3c45)CCCC2,1.0,1,0.645614,1,0.946290,1,0.687504,1,0.873967,1,1.000000,1
2,AMES_2,O=c1c2ccccc2c(=O)c2c1ccc1c2[nH]c2c3c(=O)c4cccc...,0.0,1,0.768421,1,0.723822,1,0.819435,1,0.501393,1,0.714286,1
3,AMES_3,[N-]=[N+]=CC(=O)NCC(=O)NN,1.0,1,0.709649,1,0.887996,1,0.775291,1,0.806515,1,0.857143,1
4,AMES_4,[N-]=[N+]=C1C=NC(=O)NC1=O,1.0,1,0.698538,1,0.867479,1,0.814319,1,0.908299,1,0.714286,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6439,AMES_6439,CC(C)=C1CCC(C)CC1=O,0.0,1,0.774935,1,0.735004,1,0.659195,1,0.830495,1,0.857143,1
6440,AMES_6440,CC(CCc1ccccc1)c1ccccc1,0.0,1,0.822807,1,0.754054,1,0.646190,1,0.575176,1,1.000000,1
6441,AMES_6441,CCOP(=S)(CC)Sc1ccccc1,0.0,1,0.968421,1,0.993009,1,0.814835,1,0.916946,1,1.000000,1
6442,AMES_6442,C=C(C)C1CC=C(C)C(OC(C)=O)C1,0.0,1,0.796959,1,0.768768,1,0.700767,1,0.890627,1,0.857143,1


In [15]:
y_pred_ave = df_preds["mode"]

conf_matrix_ave = confusion_matrix(y_real, y_pred_ave) 
TP_ave = conf_matrix_ave[1][1]
TN_ave = conf_matrix_ave[0][0]
FP_ave = conf_matrix_ave[0][1] 
FN_ave = conf_matrix_ave[1][0]
Accuracy_ave = accuracy_score(y_real, y_pred_ave)
Precision_ave = precision_score(y_real, y_pred_ave)
Sensitivity_ave = recall_score(y_real, y_pred_ave)
Specificity_ave = round( TN_ave / (TN_ave+FP_ave),4 )
f1_scores_ave = f1_score(y_real, y_pred_ave)
f1_scores_W_ave = f1_score(y_real, y_pred_ave, average="weighted")
f1_scores_M_ave = f1_score(y_real, y_pred_ave, average="macro")
BA_scores_ave = balanced_accuracy_score(y_real, y_pred_ave)
MCC_ave= matthews_corrcoef(y_real, y_pred_ave)
NPV_ave= round( TN_ave / (TN_ave+FN_ave),4 )
ROC_AUC_ave = roc_auc_score(y_real, y_pred_ave)

ave = pd.DataFrame({  'Combined Models':[np.mean(TP_ave), np.mean(TN_ave),
                                    np.mean(FP_ave), np.mean(FN_ave),
                                    np.mean(Accuracy_ave),
                                    np.mean(Precision_ave),
                                    np.mean(Sensitivity_ave),
                                    np.mean(Specificity_ave),
                                    np.mean(f1_scores_ave),
                                    np.mean(f1_scores_W_ave), 
                                    np.mean(f1_scores_M_ave), 
                                    np.mean(BA_scores_ave), 
                                    np.mean(MCC_ave),
                                    np.mean(NPV_ave),
                                    np.mean(ROC_AUC_ave)],
                 }) 

mat_met['Combined Models'] = ave  
print(mat_met)

                 Metric           rf         lgbm          svm        ridge  \
0                    TP  2710.000000  2859.000000  2910.000000  2920.000000   
1                    TN   875.000000   833.000000   820.000000   823.000000   
2                    FP  1915.000000  1957.000000  1970.000000  1967.000000   
3                    FN   944.000000   795.000000   744.000000   734.000000   
4              Accuracy     0.556331     0.572936     0.578833     0.580850   
5             Precision     0.585946     0.593646     0.596311     0.597504   
6           Sensitivity     0.741653     0.782430     0.796388     0.799124   
7           Specificity     0.313600     0.298600     0.293900     0.295000   
8              F1 score     0.654668     0.675089     0.681978     0.683761   
9   F1 score (weighted)     0.535614     0.546068     0.549789     0.551661   
10     F1 score (macro)     0.517180     0.526091     0.529322     0.531206   
11    Balanced Accuracy     0.527637     0.540498   

In [16]:
#df_preds.to_csv("input/External_FTO_IC50_singleProtein_RelationEqual_Predictions.csv", index=False)

In [17]:
#mat_met.to_csv("Statistics_ExternalSet_RelationEqual.csv",index=False)

In [18]:
mat_met.to_csv("Evaluation_tdc_ames.csv",index=False)

In [19]:
df_preds.sort_values(["pred_rf_proba"], ascending=False)

,Name,SMILES,Toxicity,pred_rf,pred_rf_proba,pred_lgbm,pred_lgbm_proba,pred_svm,pred_svm_proba,pred_ridge,pred_ridge_proba,pred_knn,pred_knn_proba,mode
5247,AMES_5247,c1ccc(-c2ccc(-c3ccccc3)cc2)cc1,0.0,1,1.000000,1,0.972494,1,0.918278,1,0.826080,1,1.0,1
4763,AMES_4763,CC(Cl)C(Br)CBr,1.0,1,1.000000,1,0.988481,1,0.882966,1,0.945357,1,1.0,1
5345,AMES_5345,CC/C=C/C=O,1.0,1,1.000000,1,0.989540,1,0.961304,1,0.933090,1,1.0,1
4967,AMES_4967,c1ccc2c3c(ccc2c1)-c1cccc2cccc-3c12,1.0,1,1.000000,1,0.947347,1,0.870313,1,0.772603,1,1.0,1
1328,AMES_1328,O=C(Cl)c1c(Cl)cccc1Cl,1.0,1,1.000000,1,0.979541,1,0.948004,1,0.896746,1,1.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1202,AMES_1202,Cc1cccc(OC[C@@H](O)CNC(C)(C)C)c1C,0.0,0,0.005263,0,0.038335,0,0.104967,0,0.237941,0,0.0,0
6284,AMES_6284,Cc1cccc(OCC(O)CNC(C)(C)C)c1C,0.0,0,0.005263,0,0.038335,0,0.104967,0,0.237941,0,0.0,0
1368,AMES_1368,C=CCc1ccccc1OCC(O)CNC(C)C,0.0,0,0.003509,0,0.016854,0,0.059201,0,0.283375,0,0.0,0
4085,AMES_4085,O=C(CCCN1CCC(O)(c2ccc(Cl)cc2)CC1)c1ccc(F)cc1,0.0,0,0.003509,0,0.045237,0,0.148642,0,0.118512,0,0.0,0


In [20]:
df_toxic = df_preds[df_preds["Toxicity"] ==1.0]
df_toxic

,Name,SMILES,Toxicity,pred_rf,pred_rf_proba,pred_lgbm,pred_lgbm_proba,pred_svm,pred_svm_proba,pred_ridge,pred_ridge_proba,pred_knn,pred_knn_proba,mode
0,AMES_0,O=[N+]([O-])c1ccc2ccc3ccc([N+](=O)[O-])c4c5ccc...,1.0,1,0.680702,1,0.976062,1,0.836607,1,0.899359,1,1.000000,1
1,AMES_1,O=[N+]([O-])c1c2c(c3ccc4cccc5ccc1c3c45)CCCC2,1.0,1,0.645614,1,0.946290,1,0.687504,1,0.873967,1,1.000000,1
3,AMES_3,[N-]=[N+]=CC(=O)NCC(=O)NN,1.0,1,0.709649,1,0.887996,1,0.775291,1,0.806515,1,0.857143,1
4,AMES_4,[N-]=[N+]=C1C=NC(=O)NC1=O,1.0,1,0.698538,1,0.867479,1,0.814319,1,0.908299,1,0.714286,1
5,AMES_5,[N-]=[N+]=CC(=O)NCC(N)=O,1.0,1,0.743860,1,0.827306,1,0.822771,1,0.830067,1,0.857143,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6430,AMES_6430,c1cc2c(cc1CC1CO1)OCO2,1.0,1,0.910526,1,0.935968,1,0.883203,1,0.849419,1,1.000000,1
6431,AMES_6431,N=c1ccn(C2CCC(CO)O2)c(=O)[nH]1,1.0,0,0.314035,0,0.292646,0,0.414940,0,0.471086,0,0.142857,0
6435,AMES_6435,CC1CCC(C(C)(C)OO)CC1,1.0,1,0.779942,1,0.797930,1,0.900721,1,0.634269,1,0.714286,1
6436,AMES_6436,ClCC1(C(Cl)Cl)C2CC(Cl)(Cl)C1(CCl)C(Cl)C2Cl,1.0,1,0.917135,1,0.945818,1,0.903035,1,0.806410,1,0.857143,1


In [21]:
df_pred_toxic_top = df_toxic[(df_toxic["pred_rf_proba"] >=0.8) & (df_toxic["pred_lgbm_proba"]  >=0.8) & (df_toxic["pred_svm_proba"]  >=0.8) & 
(df_toxic["pred_ridge_proba"]  >=0.8) & (df_toxic["pred_knn_proba"]  >=0.8)  ]
df_pred_toxic_top

,Name,SMILES,Toxicity,pred_rf,pred_rf_proba,pred_lgbm,pred_lgbm_proba,pred_svm,pred_svm_proba,pred_ridge,pred_ridge_proba,pred_knn,pred_knn_proba,mode
7,AMES_7,[N-]=[N+]=CC(=O)OCC(N)C(=O)O,1.0,1,0.811696,1,0.905512,1,0.877476,1,0.879903,1,0.857143,1
12,AMES_12,Cc1cccc([N+](=O)[O-])c1C,1.0,1,0.947368,1,0.993041,1,0.867601,1,0.940560,1,1.000000,1
33,AMES_33,CCN=NNCC,1.0,1,0.835380,1,0.939146,1,0.934044,1,0.838953,1,1.000000,1
34,AMES_34,O=[N+]([O-])c1cccc(O)c1,1.0,1,0.857895,1,0.988653,1,0.886149,1,0.963830,1,1.000000,1
37,AMES_37,Oc1cc2c3ccccc3ccc2c2ccccc12,1.0,1,0.971930,1,0.910966,1,0.875593,1,0.824540,1,0.857143,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6397,AMES_6397,c1cc2ccc3ccc(C4CO4)c4ccc(c1)c2c34,1.0,1,0.897243,1,0.963383,1,0.847606,1,0.816438,1,1.000000,1
6414,AMES_6414,O=NN1CCCC(Cl)C1,1.0,1,0.871930,1,0.992746,1,0.903402,1,0.840214,1,1.000000,1
6418,AMES_6418,c1ccc2c(c1)ccc1c3ccccc3c(C3CO3)cc21,1.0,1,0.897243,1,0.963383,1,0.847606,1,0.816438,1,1.000000,1
6430,AMES_6430,c1cc2c(cc1CC1CO1)OCO2,1.0,1,0.910526,1,0.935968,1,0.883203,1,0.849419,1,1.000000,1


In [22]:
import mols2grid

In [23]:
mols2grid.display(df_pred_toxic_top,smiles_col="SMILES",subset=["img","Name","Toxicity", "pred_ridge_proba", ],selection=False, addStereoAnnotation=True, size=(300, 250) )

MolGridWidget()

In [24]:
df_pred_toxic_last = df_toxic[(df_toxic["pred_rf_proba"] <=0.2) & (df_toxic["pred_lgbm_proba"]  <=0.2) & (df_toxic["pred_svm_proba"]  <=0.2)
& (df_toxic["pred_ridge_proba"]  <=0.2) & (df_toxic["pred_ridge_proba"]  <=0.2) ]
df_pred_toxic_last

,Name,SMILES,Toxicity,pred_rf,pred_rf_proba,pred_lgbm,pred_lgbm_proba,pred_svm,pred_svm_proba,pred_ridge,pred_ridge_proba,pred_knn,pred_knn_proba,mode
51,AMES_51,CN(C)CCNC(=O)c1cccc2c1C(=O)c1ccccc1C2=O,1.0,0,0.114620,0,0.053983,0,0.173675,0,0.053430,0,0.000000,0
66,AMES_66,COc1cccc2c1C(=O)c1c(O)c3c(c(O)c1C2=O)CC(O)(C(=...,1.0,0,0.088889,0,0.171931,0,0.157274,0,0.142145,0,0.285714,0
73,AMES_73,CCN(CC)CCNc1ccc2nc(C)n3c4ccc(O)cc4c(=O)c1c23,1.0,0,0.114474,0,0.046978,0,0.113057,0,0.075383,0,0.000000,0
206,AMES_206,CC[N+]([O-])(CC)CCNc1ccc(CO)c2sc3ccccc3c(=O)c12,1.0,0,0.128070,0,0.036501,0,0.128353,0,0.040270,0,0.142857,0
222,AMES_222,CCC1(O)CC(OC2CC(N(C)C)C(OC3CC(O)C(O)C(C)O3)C(C...,1.0,0,0.179298,0,0.096065,0,0.092730,0,0.168937,0,0.142857,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5953,AMES_5953,CC1COc2c(N3CCN(C)CC3)c(F)c(C(=O)O)c3c(=O)ccn1c23,1.0,0,0.138830,0,0.111695,0,0.195905,0,0.080026,0,0.142857,0
5968,AMES_5968,CN(C)CCNC(=O)n1c2ccccc2c(=O)c2ccccc21,1.0,0,0.075146,0,0.015697,0,0.119571,0,0.066940,0,0.000000,0
6015,AMES_6015,CCN(CC)CCCC(C)Nc1ccnc2cc(Cl)ccc12,1.0,0,0.053509,0,0.017987,0,0.158066,0,0.110733,0,0.000000,0
6025,AMES_6025,CCn1cc(C(=O)O)c(=O)c2cc(F)c(N3CCNCC3)cc21,1.0,0,0.153216,0,0.032757,0,0.070081,0,0.024571,0,0.142857,0


In [25]:
mols2grid.display(df_pred_toxic_last,smiles_col="SMILES",subset=["img","Name","Toxicity", "pred_ridge_proba"],selection=False, addStereoAnnotation=True, size=(300, 250))

MolGridWidget()

In [26]:
df_nontoxic= df_preds[df_preds["Toxicity"] == 0.0]
df_nontoxic

,Name,SMILES,Toxicity,pred_rf,pred_rf_proba,pred_lgbm,pred_lgbm_proba,pred_svm,pred_svm_proba,pred_ridge,pred_ridge_proba,pred_knn,pred_knn_proba,mode
2,AMES_2,O=c1c2ccccc2c(=O)c2c1ccc1c2[nH]c2c3c(=O)c4cccc...,0.0,1,0.768421,1,0.723822,1,0.819435,1,0.501393,1,0.714286,1
9,AMES_9,CC(=O)OC1(C(C)=O)CCC2C3C=C(Cl)C4=CC(=O)OCC4(C)...,0.0,0,0.289177,0,0.268866,0,0.346554,0,0.430946,0,0.428571,0
11,AMES_11,CC(C)CC(=O)Nc1snc2ccccc12,0.0,0,0.392982,0,0.199915,0,0.329353,0,0.425439,1,0.714286,0
13,AMES_13,C#CC1(OC(=O)CCCCCC)CCC2C3CCC4=CC(=O)CCC4C3CCC21C,0.0,0,0.014737,0,0.168644,0,0.120754,0,0.261452,0,0.000000,0
14,AMES_14,CCC[N+](=O)[O-],0.0,1,0.821053,1,0.966367,1,0.928609,1,0.942622,1,0.857143,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6439,AMES_6439,CC(C)=C1CCC(C)CC1=O,0.0,1,0.774935,1,0.735004,1,0.659195,1,0.830495,1,0.857143,1
6440,AMES_6440,CC(CCc1ccccc1)c1ccccc1,0.0,1,0.822807,1,0.754054,1,0.646190,1,0.575176,1,1.000000,1
6441,AMES_6441,CCOP(=S)(CC)Sc1ccccc1,0.0,1,0.968421,1,0.993009,1,0.814835,1,0.916946,1,1.000000,1
6442,AMES_6442,C=C(C)C1CC=C(C)C(OC(C)=O)C1,0.0,1,0.796959,1,0.768768,1,0.700767,1,0.890627,1,0.857143,1


In [27]:
df_nontoxic_top = df_nontoxic[(df_nontoxic["pred_rf_proba"] <=0.2) & (df_nontoxic["pred_lgbm_proba"]  <=0.2) & (df_nontoxic["pred_svm_proba"]  <=0.2)
& (df_nontoxic["pred_ridge_proba"]  <=0.2) & (df_nontoxic["pred_ridge_proba"]  <=0.2) ]
df_nontoxic_top

,Name,SMILES,Toxicity,pred_rf,pred_rf_proba,pred_lgbm,pred_lgbm_proba,pred_svm,pred_svm_proba,pred_ridge,pred_ridge_proba,pred_knn,pred_knn_proba,mode
112,AMES_112,CN(c1nccc(O)n1)C1CCN(c2nc3ccccc3n2Cc2ccc(F)cc2...,0.0,0,0.031287,0,0.020888,0,0.070363,0,0.070298,0,0.000000,0
170,AMES_170,CC(C)CC(NC(=O)C(Cc1ccccc1)NC(=O)CNC(=O)CN)C(=O)O,0.0,0,0.129240,0,0.046771,0,0.075829,0,0.142849,1,0.571429,0
180,AMES_180,CNCc1ccc(CSCCN=C(NC[C@H](O)c2ccc(O)cc2)NS(C)(=...,0.0,0,0.073099,0,0.027783,0,0.103895,0,0.118737,0,0.000000,0
192,AMES_192,CN1CCN(C2=Nc3ccccc3Oc3ccc(Cl)cc32)CC1,0.0,0,0.171930,0,0.030055,0,0.116612,0,0.145279,0,0.142857,0
289,AMES_289,OC(Cn1ccnc1)c1ccc(CCc2ccccc2)cc1,0.0,0,0.166082,0,0.128069,0,0.114455,0,0.114989,0,0.000000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6206,AMES_6206,CCC1(CC(=O)O)OCCc2c1n(C)c1ccccc21,0.0,0,0.152632,0,0.158902,0,0.066026,0,0.142680,0,0.285714,0
6208,AMES_6208,CNCc1ccc(CSCCNC(=NCC(O)c2ccc(O)cc2)NS(C)(=O)=O)o1,0.0,0,0.095029,0,0.149345,0,0.156541,0,0.136485,0,0.142857,0
6275,AMES_6275,CN1CCC(CN2c3ccccc3Sc3ccccc32)C1,0.0,0,0.008187,0,0.111946,0,0.163025,0,0.120487,0,0.000000,0
6295,AMES_6295,CC1(C)SC2C(NC(=O)Cc3ccccc3)C(=O)N2C1C(=O)O,0.0,0,0.079657,0,0.125405,0,0.163100,0,0.127591,0,0.142857,0


In [28]:
mols2grid.display(df_nontoxic_top,smiles_col="SMILES",subset=["img","Name","Toxicity", "pred_ridge_proba", ],selection=False, addStereoAnnotation=True, size=(300, 250))

MolGridWidget()

In [29]:
df_nontoxic_last = df_nontoxic[(df_nontoxic["pred_rf_proba"] >=0.85) & (df_nontoxic["pred_lgbm_proba"]  >=0.85) & (df_nontoxic["pred_svm_proba"] >=0.85) 
& (df_nontoxic["pred_ridge_proba"] >=0.85) & (df_nontoxic["pred_knn_proba"]  >=0.85)]
df_nontoxic_last

,Name,SMILES,Toxicity,pred_rf,pred_rf_proba,pred_lgbm,pred_lgbm_proba,pred_svm,pred_svm_proba,pred_ridge,pred_ridge_proba,pred_knn,pred_knn_proba,mode
47,AMES_47,CCSCCSP(=O)(OC)OC,0.0,1,0.975439,1,0.983840,1,0.936515,1,0.905827,1,1.000000,1
58,AMES_58,O=C(O)c1ccccc1,0.0,1,0.866667,1,0.883624,1,0.892785,1,0.897024,1,0.857143,1
95,AMES_95,C1CCCC2OC2CC1,0.0,1,0.951462,1,0.979264,1,0.938397,1,0.939240,1,1.000000,1
110,AMES_110,CCCCC=O,0.0,1,0.992982,1,0.977139,1,0.954006,1,0.887707,1,1.000000,1
117,AMES_117,Cc1ccc([N+](=O)[O-])cc1S(=O)(=O)O,0.0,1,0.859649,1,0.991756,1,0.918334,1,0.937011,1,1.000000,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6243,AMES_6243,O=S(=O)(O)c1cc(S(=O)(=O)O)c2c(N=Nc3ccccc3)c(O)...,0.0,1,1.000000,1,0.981892,1,0.893201,1,0.857945,1,1.000000,1
6268,AMES_6268,C1=CC2C3C=CC(C3)C2C1,0.0,1,0.991228,1,0.987348,1,0.970095,1,0.901636,1,1.000000,1
6269,AMES_6269,CC1=CCC2CC1C2(C)C,0.0,1,0.964160,1,0.924687,1,0.891084,1,0.868507,1,1.000000,1
6422,AMES_6422,C=C1C(=O)OC2CC(C)C3C=CC(=O)C3(C)C(O)C12,0.0,1,0.913099,1,0.914039,1,0.853923,1,0.898317,1,0.857143,1


In [30]:
mols2grid.display(df_nontoxic_last,smiles_col="SMILES",subset=["img","Name","Toxicity", "pred_ridge_proba", ],selection=False, addStereoAnnotation=True, size=(300, 250))

MolGridWidget()

In [31]:
mat_met


,Metric,rf,lgbm,svm,ridge,knn,Combined Models
0,TP,2710.000000,2859.000000,2910.000000,2920.000000,2995.000000,2945.000000
1,TN,875.000000,833.000000,820.000000,823.000000,793.000000,807.000000
2,FP,1915.000000,1957.000000,1970.000000,1967.000000,1997.000000,1983.000000
3,FN,944.000000,795.000000,744.000000,734.000000,659.000000,709.000000
4,Accuracy,0.556331,0.572936,0.578833,0.580850,0.587834,0.582247
5,Precision,0.585946,0.593646,0.596311,0.597504,0.599960,0.597606
6,Sensitivity,0.741653,0.782430,0.796388,0.799124,0.819650,0.805966
7,Specificity,0.313600,0.298600,0.293900,0.295000,0.284200,0.289200
8,F1 score,0.654668,0.675089,0.681978,0.683761,0.692806,0.686320
9,F1 score (weighted),0.535614,0.546068,0.549789,0.551661,0.554724,0.551455


### DILI Set

In [34]:
df = pd.read_pickle('tdc_dili_FP.csv')
print(len(df))
df.head()

404


,Name,SMILES,fp_MACCS,Toxicity
0,DILI_0,CC(=O)OCC[N+](C)(C)C,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0.0
1,DILI_1,C[N+](C)(C)CC(=O)[O-],"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0.0
2,DILI_2,O=C(NC(CO)C(O)c1ccc([N+](=O)[O-])cc1)C(Cl)Cl,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0.0
3,DILI_3,O=C(O)c1ccccc1O,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0.0
4,DILI_4,CC(NC(C)(C)C)C(=O)c1cccc(Cl)c1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0.0


In [35]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 404 entries, 0 to 403
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Name      404 non-null    object 
 1   SMILES    404 non-null    object 
 2   fp_MACCS  404 non-null    object 
 3   Toxicity  404 non-null    float64
dtypes: float64(1), object(3)
memory usage: 12.8+ KB


In [36]:
print("Number of toxic molecules in external set:", len(df[df["Toxicity"] == 1.0]))
print("Number of nontoxic molecules in external set:", len(df[df["Toxicity"] == 0.0]))

Number of toxic molecules in external set: 193
Number of nontoxic molecules in external set: 211


#### Test Each Model One by One

In [37]:
#By using Morgan fingerprints with radius of 3 and 1025 bits
X_internal = np.array(list((df_internal['fp_MACCS']))).astype(float)
print(X_internal.shape)
y_internal=df_internal.Toxicity
print(y_internal.head())

(6403, 167)
0    0
1    0
2    0
3    0
4    0
Name: Toxicity, dtype: int64


In [38]:
#By using Morgan fingerprints with radius of 3 and 1025 bits
X = np.array(list((df['fp_MACCS']))).astype(float)
print(X.shape)
y_real=df.Toxicity
print(y_real.head())

(404, 167)
0    0.0
1    0.0
2    0.0
3    0.0
4    0.0
Name: Toxicity, dtype: float64


In [39]:
optimizedCV_clf_rf =  joblib.load("FinalModels/RF_optimized_full.joblib")
y_pred_rf = optimizedCV_clf_rf.predict(X)

#add to a dataframe
df_preds = df[["Name", "SMILES", "Toxicity"]]
df_preds["pred_rf"] = y_pred_rf
df_preds["pred_rf_proba"] = optimizedCV_clf_rf.predict_proba(X).tolist()
df_preds["pred_rf_proba"] = df_preds["pred_rf_proba"].str[1]

conf_matrix_rf = confusion_matrix(y_real, y_pred_rf) 
TP_rf = conf_matrix_rf[1][1]
TN_rf = conf_matrix_rf[0][0]
FP_rf = conf_matrix_rf[0][1] 
FN_rf = conf_matrix_rf[1][0]
Accuracy_rf = accuracy_score(y_real, y_pred_rf)
Precision_rf = precision_score(y_real, y_pred_rf)
Sensitivity_rf = recall_score(y_real, y_pred_rf)
Specificity_rf = round( TN_rf / (TN_rf+FP_rf),4 )
f1_scores_rf = f1_score(y_real, y_pred_rf)
f1_scores_W_rf = f1_score(y_real, y_pred_rf, average="weighted")
f1_scores_M_rf = f1_score(y_real, y_pred_rf, average="macro")
BA_scores_rf = balanced_accuracy_score(y_real, y_pred_rf)
MCC_rf= matthews_corrcoef(y_real, y_pred_rf)
NPV_rf= round( TN_rf / (TN_rf+FN_rf),4 )
ROC_AUC_rf = roc_auc_score(y_real, y_pred_rf)

mat_met = pd.DataFrame({'Metric':['TP','TN','FP','FN','Accuracy','Precision','Sensitivity','Specificity','F1 score','F1 score (weighted)',
                            'F1 score (macro)', 'Balanced Accuracy','MCC','NPV','ROC_AUC'],     
                        'rf':[np.mean(TP_rf), np.mean(TN_rf),
                                    np.mean(FP_rf), np.mean(FN_rf),
                                    np.mean(Accuracy_rf),
                                    np.mean(Precision_rf),
                                    np.mean(Sensitivity_rf),
                                    np.mean(Specificity_rf),
                                    np.mean(f1_scores_rf),
                                    np.mean(f1_scores_W_rf), 
                                    np.mean(f1_scores_M_rf), 
                                    np.mean(BA_scores_rf), 
                                    np.mean(MCC_rf),
                                    np.mean(NPV_rf),
                                    np.mean(ROC_AUC_rf)],
                   }) 
    
print(mat_met)

                 Metric          rf
0                    TP   40.000000
1                    TN  164.000000
2                    FP   47.000000
3                    FN  153.000000
4              Accuracy    0.504950
5             Precision    0.459770
6           Sensitivity    0.207254
7           Specificity    0.777300
8              F1 score    0.285714
9   F1 score (weighted)    0.460937
10     F1 score (macro)    0.453463
11    Balanced Accuracy    0.492253
12                  MCC   -0.018829
13                  NPV    0.517400
14              ROC_AUC    0.492253


In [40]:
optimizedCV_clf_lgbm =  joblib.load("FinalModels/LGBM_optimized_full.joblib")
y_pred_lgbm = optimizedCV_clf_lgbm.predict(X)
df_preds["pred_lgbm"] = y_pred_lgbm
df_preds["pred_lgbm_proba"] = optimizedCV_clf_lgbm.predict_proba(X).tolist()
df_preds["pred_lgbm_proba"] = df_preds["pred_lgbm_proba"].str[1]

conf_matrix_lgbm = confusion_matrix(y_real, y_pred_lgbm) 
TP_lgbm = conf_matrix_lgbm[1][1]
TN_lgbm = conf_matrix_lgbm[0][0]
FP_lgbm = conf_matrix_lgbm[0][1] 
FN_lgbm = conf_matrix_lgbm[1][0]
Accuracy_lgbm = accuracy_score(y_real, y_pred_lgbm)
Precision_lgbm = precision_score(y_real, y_pred_lgbm)
Sensitivity_lgbm = recall_score(y_real, y_pred_lgbm)
Specificity_lgbm = round( TN_lgbm / (TN_lgbm+FP_lgbm),4 )
f1_scores_lgbm = f1_score(y_real, y_pred_lgbm)
f1_scores_W_lgbm = f1_score(y_real, y_pred_lgbm, average="weighted")
f1_scores_M_lgbm = f1_score(y_real, y_pred_lgbm, average="macro")
BA_scores_lgbm = balanced_accuracy_score(y_real, y_pred_lgbm)
MCC_lgbm= matthews_corrcoef(y_real, y_pred_lgbm)
NPV_lgbm= round( TN_lgbm / (TN_lgbm+FN_lgbm),4 )
ROC_AUC_lgbm = roc_auc_score(y_real, y_pred_lgbm)

lgbm = pd.DataFrame({  'lgbm':[np.mean(TP_lgbm), np.mean(TN_lgbm),
                                    np.mean(FP_lgbm), np.mean(FN_lgbm),
                                    np.mean(Accuracy_lgbm),
                                    np.mean(Precision_lgbm),
                                    np.mean(Sensitivity_lgbm),
                                    np.mean(Specificity_lgbm),
                                    np.mean(f1_scores_lgbm),
                                    np.mean(f1_scores_W_lgbm), 
                                    np.mean(f1_scores_M_lgbm), 
                                    np.mean(BA_scores_lgbm), 
                                    np.mean(MCC_lgbm),
                                    np.mean(NPV_lgbm),
                                    np.mean(ROC_AUC_lgbm)],
                 }) 

mat_met['lgbm'] = lgbm  
    
print(mat_met)

                 Metric          rf        lgbm
0                    TP   40.000000   51.000000
1                    TN  164.000000  168.000000
2                    FP   47.000000   43.000000
3                    FN  153.000000  142.000000
4              Accuracy    0.504950    0.542079
5             Precision    0.459770    0.542553
6           Sensitivity    0.207254    0.264249
7           Specificity    0.777300    0.796200
8              F1 score    0.285714    0.355401
9   F1 score (weighted)    0.460937    0.506607
10     F1 score (macro)    0.453463    0.500157
11    Balanced Accuracy    0.492253    0.530229
12                  MCC   -0.018829    0.071470
13                  NPV    0.517400    0.541900
14              ROC_AUC    0.492253    0.530229


In [41]:
optimizedCV_clf_svm =  joblib.load("FinalModels/SVC_optimized_full.joblib")
y_pred_svm = optimizedCV_clf_svm.predict(X)
df_preds["pred_svm"] = y_pred_svm
df_preds["pred_svm_proba"] = optimizedCV_clf_svm.predict_proba(X).tolist()
df_preds["pred_svm_proba"] = df_preds["pred_svm_proba"].str[1]

conf_matrix_svm = confusion_matrix(y_real, y_pred_svm) 
TP_svm = conf_matrix_svm[1][1]
TN_svm = conf_matrix_svm[0][0]
FP_svm = conf_matrix_svm[0][1] 
FN_svm = conf_matrix_svm[1][0]
Accuracy_svm = accuracy_score(y_real, y_pred_svm)
Precision_svm = precision_score(y_real, y_pred_svm)
Sensitivity_svm = recall_score(y_real, y_pred_svm)
Specificity_svm = round( TN_svm / (TN_svm+FP_svm),4 )
f1_scores_svm = f1_score(y_real, y_pred_svm)
f1_scores_W_svm = f1_score(y_real, y_pred_svm, average="weighted")
f1_scores_M_svm = f1_score(y_real, y_pred_svm, average="macro")
BA_scores_svm = balanced_accuracy_score(y_real, y_pred_svm)
MCC_svm= matthews_corrcoef(y_real, y_pred_svm)
NPV_svm= round( TN_svm / (TN_svm+FN_svm),4 )
ROC_AUC_svm = roc_auc_score(y_real, y_pred_svm)

svm = pd.DataFrame({  'svm':[np.mean(TP_svm), np.mean(TN_svm),
                                    np.mean(FP_svm), np.mean(FN_svm),
                                    np.mean(Accuracy_svm),
                                    np.mean(Precision_svm),
                                    np.mean(Sensitivity_svm),
                                    np.mean(Specificity_svm),
                                    np.mean(f1_scores_svm),
                                    np.mean(f1_scores_W_svm), 
                                    np.mean(f1_scores_M_svm), 
                                    np.mean(BA_scores_svm), 
                                    np.mean(MCC_svm),
                                    np.mean(NPV_svm),
                                    np.mean(ROC_AUC_svm)],
                 }) 

mat_met['svm'] = svm  
print(mat_met)

                 Metric          rf        lgbm         svm
0                    TP   40.000000   51.000000   49.000000
1                    TN  164.000000  168.000000  159.000000
2                    FP   47.000000   43.000000   52.000000
3                    FN  153.000000  142.000000  144.000000
4              Accuracy    0.504950    0.542079    0.514851
5             Precision    0.459770    0.542553    0.485149
6           Sensitivity    0.207254    0.264249    0.253886
7           Specificity    0.777300    0.796200    0.753600
8              F1 score    0.285714    0.355401    0.333333
9   F1 score (weighted)    0.460937    0.506607    0.482362
10     F1 score (macro)    0.453463    0.500157    0.476005
11    Balanced Accuracy    0.492253    0.530229    0.503720
12                  MCC   -0.018829    0.071470    0.008583
13                  NPV    0.517400    0.541900    0.524800
14              ROC_AUC    0.492253    0.530229    0.503720


In [42]:
optimizedCV_clf_ridge =  joblib.load("FinalModels/Ridge_optimized_full.joblib")

from sklearn.calibration import CalibratedClassifierCV
ridge_cal = CalibratedClassifierCV(
    optimizedCV_clf_ridge,
    method="sigmoid",   # Platt scaling
    cv="prefit"
)

ridge_cal.fit(X_internal, y_internal)

y_pred_ridge = ridge_cal.predict(X)
df_preds["pred_ridge"] = y_pred_ridge
df_preds["pred_ridge_proba"] = ridge_cal.predict_proba(X).tolist()
df_preds["pred_ridge_proba"] = df_preds["pred_ridge_proba"].str[1]
df_preds["pred_ridge"] = y_pred_ridge
#df_preds["pred_ridge_proba"] = optimizedCV_clf_ridge.decision_function(X).tolist()
#df_preds["pred_ridge_proba"] = df_preds["pred_ridge_proba"].str[1]

conf_matrix_ridge = confusion_matrix(y_real, y_pred_ridge) 
TP_ridge = conf_matrix_ridge[1][1]
TN_ridge = conf_matrix_ridge[0][0]
FP_ridge = conf_matrix_ridge[0][1] 
FN_ridge = conf_matrix_ridge[1][0]
Accuracy_ridge = accuracy_score(y_real, y_pred_ridge)
Precision_ridge = precision_score(y_real, y_pred_ridge)
Sensitivity_ridge = recall_score(y_real, y_pred_ridge)
Specificity_ridge = round( TN_ridge / (TN_ridge+FP_ridge),4 )
f1_scores_ridge = f1_score(y_real, y_pred_ridge)
f1_scores_W_ridge = f1_score(y_real, y_pred_ridge, average="weighted")
f1_scores_M_ridge = f1_score(y_real, y_pred_ridge, average="macro")
BA_scores_ridge = balanced_accuracy_score(y_real, y_pred_ridge)
MCC_ridge= matthews_corrcoef(y_real, y_pred_ridge)
NPV_ridge= round( TN_ridge / (TN_ridge+FN_ridge),4 )
ROC_AUC_ridge = ROC_AUC_ridge = roc_auc_score(y_real,
    optimizedCV_clf_ridge.decision_function(X))

ridge = pd.DataFrame({  'ridge':[np.mean(TP_ridge), np.mean(TN_ridge),
                                    np.mean(FP_ridge), np.mean(FN_ridge),
                                    np.mean(Accuracy_ridge),
                                    np.mean(Precision_ridge),
                                    np.mean(Sensitivity_ridge),
                                    np.mean(Specificity_ridge),
                                    np.mean(f1_scores_ridge),
                                    np.mean(f1_scores_W_ridge), 
                                    np.mean(f1_scores_M_ridge), 
                                    np.mean(BA_scores_ridge), 
                                    np.mean(MCC_ridge),
                                    np.mean(NPV_ridge),
                                    np.mean(ROC_AUC_ridge)],
                 }) 

mat_met['ridge'] = ridge  
print(mat_met)

                 Metric          rf        lgbm         svm       ridge
0                    TP   40.000000   51.000000   49.000000   57.000000
1                    TN  164.000000  168.000000  159.000000  159.000000
2                    FP   47.000000   43.000000   52.000000   52.000000
3                    FN  153.000000  142.000000  144.000000  136.000000
4              Accuracy    0.504950    0.542079    0.514851    0.534653
5             Precision    0.459770    0.542553    0.485149    0.522936
6           Sensitivity    0.207254    0.264249    0.253886    0.295337
7           Specificity    0.777300    0.796200    0.753600    0.753600
8              F1 score    0.285714    0.355401    0.333333    0.377483
9   F1 score (weighted)    0.460937    0.506607    0.482362    0.508562
10     F1 score (macro)    0.453463    0.500157    0.476005    0.502971
11    Balanced Accuracy    0.492253    0.530229    0.503720    0.524446
12                  MCC   -0.018829    0.071470    0.008583    0

In [43]:
optimizedCV_clf_knn =  joblib.load("FinalModels/KNN_optimized_full.joblib")
y_pred_knn = optimizedCV_clf_knn.predict(X)
df_preds["pred_knn"] = y_pred_knn
df_preds["pred_knn_proba"] = optimizedCV_clf_knn.predict_proba(X).tolist()
df_preds["pred_knn_proba"] = df_preds["pred_knn_proba"].str[1]

conf_matrix_knn = confusion_matrix(y_real, y_pred_knn) 
TP_knn = conf_matrix_knn[1][1]
TN_knn = conf_matrix_knn[0][0]
FP_knn = conf_matrix_knn[0][1] 
FN_knn = conf_matrix_knn[1][0]
Accuracy_knn = accuracy_score(y_real, y_pred_knn)
Precision_knn = precision_score(y_real, y_pred_knn)
Sensitivity_knn = recall_score(y_real, y_pred_knn)
Specificity_knn = round( TN_knn / (TN_knn+FP_knn),4 )
f1_scores_knn = f1_score(y_real, y_pred_knn)
f1_scores_W_knn = f1_score(y_real, y_pred_knn, average="weighted")
f1_scores_M_knn = f1_score(y_real, y_pred_knn, average="macro")
BA_scores_knn = balanced_accuracy_score(y_real, y_pred_knn)
MCC_knn= matthews_corrcoef(y_real, y_pred_knn)
NPV_knn= round( TN_knn / (TN_knn+FN_knn),4 )
ROC_AUC_knn = roc_auc_score(y_real, y_pred_knn)

knn = pd.DataFrame({  'knn':[np.mean(TP_knn), np.mean(TN_knn),
                                    np.mean(FP_knn), np.mean(FN_knn),
                                    np.mean(Accuracy_knn),
                                    np.mean(Precision_knn),
                                    np.mean(Sensitivity_knn),
                                    np.mean(Specificity_knn),
                                    np.mean(f1_scores_knn),
                                    np.mean(f1_scores_W_knn), 
                                    np.mean(f1_scores_M_knn), 
                                    np.mean(BA_scores_knn), 
                                    np.mean(MCC_knn),
                                    np.mean(NPV_knn),
                                    np.mean(ROC_AUC_knn)],
                 }) 

mat_met['knn'] = knn  
print(mat_met)

                 Metric          rf        lgbm         svm       ridge  \
0                    TP   40.000000   51.000000   49.000000   57.000000   
1                    TN  164.000000  168.000000  159.000000  159.000000   
2                    FP   47.000000   43.000000   52.000000   52.000000   
3                    FN  153.000000  142.000000  144.000000  136.000000   
4              Accuracy    0.504950    0.542079    0.514851    0.534653   
5             Precision    0.459770    0.542553    0.485149    0.522936   
6           Sensitivity    0.207254    0.264249    0.253886    0.295337   
7           Specificity    0.777300    0.796200    0.753600    0.753600   
8              F1 score    0.285714    0.355401    0.333333    0.377483   
9   F1 score (weighted)    0.460937    0.506607    0.482362    0.508562   
10     F1 score (macro)    0.453463    0.500157    0.476005    0.502971   
11    Balanced Accuracy    0.492253    0.530229    0.503720    0.524446   
12                  MCC  

In [44]:
df_preds

,Name,SMILES,Toxicity,pred_rf,pred_rf_proba,pred_lgbm,pred_lgbm_proba,pred_svm,pred_svm_proba,pred_ridge,pred_ridge_proba,pred_knn,pred_knn_proba
0,DILI_0,CC(=O)OCC[N+](C)(C)C,0.0,1,0.836790,1,0.718751,1,0.666596,1,0.527748,0,0.454545
1,DILI_1,C[N+](C)(C)CC(=O)[O-],0.0,1,0.752593,1,0.861657,1,0.807273,1,0.710851,1,0.772727
2,DILI_2,O=C(NC(CO)C(O)c1ccc([N+](=O)[O-])cc1)C(Cl)Cl,0.0,0,0.454074,1,0.648162,1,0.760184,1,0.918027,0,0.500000
3,DILI_3,O=C(O)c1ccccc1O,0.0,1,0.832593,1,0.646671,1,0.795398,1,0.800162,1,0.636364
4,DILI_4,CC(NC(C)(C)C)C(=O)c1cccc(Cl)c1,0.0,1,0.722840,1,0.760705,1,0.699934,1,0.667100,1,0.818182
...,...,...,...,...,...,...,...,...,...,...,...,...,...
399,DILI_399,Cc1ccc(Nc2nccc(N(C)c3ccc4c(C)n(C)nc4c3)n2)cc1S...,1.0,0,0.274691,0,0.090602,0,0.309576,0,0.331919,0,0.181818
400,DILI_400,CC1C=CC=CCCC=CC=CC=CC=CC(OC2OC(C)C(O)C(N)C2O)C...,0.0,0,0.115062,0,0.184623,0,0.243432,0,0.325142,0,0.318182
401,DILI_401,C=C1c2cccc(O)c2C(O)=C2C(=O)C3(O)C(O)=C(C(N)=O)...,1.0,1,0.553531,1,0.526834,0,0.460178,0,0.484271,0,0.500000
402,DILI_402,O=C1OC(C(O)CO)C(O)=C1O,0.0,1,0.553390,1,0.723761,1,0.785029,1,0.809981,1,0.636364


In [45]:
df_preds["mode"] = df_preds[["pred_rf", "pred_lgbm", "pred_svm","pred_ridge", "pred_knn" ]].mode(axis=1)
df_preds

,Name,SMILES,Toxicity,pred_rf,pred_rf_proba,pred_lgbm,pred_lgbm_proba,pred_svm,pred_svm_proba,pred_ridge,pred_ridge_proba,pred_knn,pred_knn_proba,mode
0,DILI_0,CC(=O)OCC[N+](C)(C)C,0.0,1,0.836790,1,0.718751,1,0.666596,1,0.527748,0,0.454545,1
1,DILI_1,C[N+](C)(C)CC(=O)[O-],0.0,1,0.752593,1,0.861657,1,0.807273,1,0.710851,1,0.772727,1
2,DILI_2,O=C(NC(CO)C(O)c1ccc([N+](=O)[O-])cc1)C(Cl)Cl,0.0,0,0.454074,1,0.648162,1,0.760184,1,0.918027,0,0.500000,1
3,DILI_3,O=C(O)c1ccccc1O,0.0,1,0.832593,1,0.646671,1,0.795398,1,0.800162,1,0.636364,1
4,DILI_4,CC(NC(C)(C)C)C(=O)c1cccc(Cl)c1,0.0,1,0.722840,1,0.760705,1,0.699934,1,0.667100,1,0.818182,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
399,DILI_399,Cc1ccc(Nc2nccc(N(C)c3ccc4c(C)n(C)nc4c3)n2)cc1S...,1.0,0,0.274691,0,0.090602,0,0.309576,0,0.331919,0,0.181818,0
400,DILI_400,CC1C=CC=CCCC=CC=CC=CC=CC(OC2OC(C)C(O)C(N)C2O)C...,0.0,0,0.115062,0,0.184623,0,0.243432,0,0.325142,0,0.318182,0
401,DILI_401,C=C1c2cccc(O)c2C(O)=C2C(=O)C3(O)C(O)=C(C(N)=O)...,1.0,1,0.553531,1,0.526834,0,0.460178,0,0.484271,0,0.500000,0
402,DILI_402,O=C1OC(C(O)CO)C(O)=C1O,0.0,1,0.553390,1,0.723761,1,0.785029,1,0.809981,1,0.636364,1


In [46]:
y_pred_ave = df_preds["mode"]

conf_matrix_ave = confusion_matrix(y_real, y_pred_ave) 
TP_ave = conf_matrix_ave[1][1]
TN_ave = conf_matrix_ave[0][0]
FP_ave = conf_matrix_ave[0][1] 
FN_ave = conf_matrix_ave[1][0]
Accuracy_ave = accuracy_score(y_real, y_pred_ave)
Precision_ave = precision_score(y_real, y_pred_ave)
Sensitivity_ave = recall_score(y_real, y_pred_ave)
Specificity_ave = round( TN_ave / (TN_ave+FP_ave),4 )
f1_scores_ave = f1_score(y_real, y_pred_ave)
f1_scores_W_ave = f1_score(y_real, y_pred_ave, average="weighted")
f1_scores_M_ave = f1_score(y_real, y_pred_ave, average="macro")
BA_scores_ave = balanced_accuracy_score(y_real, y_pred_ave)
MCC_ave= matthews_corrcoef(y_real, y_pred_ave)
NPV_ave= round( TN_ave / (TN_ave+FN_ave),4 )
ROC_AUC_ave = roc_auc_score(y_real, y_pred_ave)

ave = pd.DataFrame({  'Combined Models':[np.mean(TP_ave), np.mean(TN_ave),
                                    np.mean(FP_ave), np.mean(FN_ave),
                                    np.mean(Accuracy_ave),
                                    np.mean(Precision_ave),
                                    np.mean(Sensitivity_ave),
                                    np.mean(Specificity_ave),
                                    np.mean(f1_scores_ave),
                                    np.mean(f1_scores_W_ave), 
                                    np.mean(f1_scores_M_ave), 
                                    np.mean(BA_scores_ave), 
                                    np.mean(MCC_ave),
                                    np.mean(NPV_ave),
                                    np.mean(ROC_AUC_ave)],
                 }) 

mat_met['Combined Models'] = ave  
print(mat_met)

                 Metric          rf        lgbm         svm       ridge  \
0                    TP   40.000000   51.000000   49.000000   57.000000   
1                    TN  164.000000  168.000000  159.000000  159.000000   
2                    FP   47.000000   43.000000   52.000000   52.000000   
3                    FN  153.000000  142.000000  144.000000  136.000000   
4              Accuracy    0.504950    0.542079    0.514851    0.534653   
5             Precision    0.459770    0.542553    0.485149    0.522936   
6           Sensitivity    0.207254    0.264249    0.253886    0.295337   
7           Specificity    0.777300    0.796200    0.753600    0.753600   
8              F1 score    0.285714    0.355401    0.333333    0.377483   
9   F1 score (weighted)    0.460937    0.506607    0.482362    0.508562   
10     F1 score (macro)    0.453463    0.500157    0.476005    0.502971   
11    Balanced Accuracy    0.492253    0.530229    0.503720    0.524446   
12                  MCC  

In [47]:
#df_preds.to_csv("input/External_FTO_IC50_singleProtein_RelationEqual_Predictions.csv", index=False)

In [48]:
#mat_met.to_csv("Statistics_ExternalSet_RelationEqual.csv",index=False)

In [49]:
mat_met.to_csv("Evaluation_tdc_dili.csv",index=False)

In [50]:
df_preds.sort_values(["pred_ridge_proba"], ascending=False)

,Name,SMILES,Toxicity,pred_rf,pred_rf_proba,pred_lgbm,pred_lgbm_proba,pred_svm,pred_svm_proba,pred_ridge,pred_ridge_proba,pred_knn,pred_knn_proba,mode
85,DILI_85,O=C1OCCN1N=Cc1ccc([N+](=O)[O-])o1,1.0,1,0.532765,1,0.873601,1,0.934271,1,0.972205,1,0.590909,1
34,DILI_34,Cn1cnc([N+](=O)[O-])c1Sc1ncnc2nc[nH]c12,1.0,0,0.417901,1,0.904264,1,0.875982,1,0.960381,0,0.500000,1
6,DILI_6,ClC1C(Cl)C(Cl)C(Cl)C(Cl)C1Cl,0.0,1,0.588102,1,0.983922,1,0.956067,1,0.951576,1,0.954545,1
128,DILI_128,COC(=O)C1=C(C)NC(C)=C(C(=O)OC)C1c1ccccc1[N+](=...,1.0,0,0.496469,1,0.963727,1,0.855297,1,0.935350,1,0.681818,1
367,DILI_367,Cc1ccc(C(=O)c2cc(O)c(O)c([N+](=O)[O-])c2)cc1,1.0,1,0.588148,1,0.936039,1,0.836207,1,0.933367,1,0.681818,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
282,DILI_282,CN(C)CCn1nnnc1SCC1=C(C(=O)O)N2C(=O)C(NC(=O)Cc3...,1.0,0,0.043210,0,0.005921,0,0.076484,0,0.022877,0,0.045455,0
319,DILI_319,CC(C)CC(NC(=O)C(Cc1ccccc1)NC(=O)c1cnccn1)B(O)O,1.0,0,0.071259,0,0.006893,0,0.026152,0,0.017827,0,0.181818,0
106,DILI_106,CCn1cc(C(=O)O)c(=O)c2cc(F)c(N3CCNC(C)C3)c(F)c21,1.0,0,0.043316,0,0.024551,0,0.034774,0,0.017673,0,0.136364,0
297,DILI_297,COCC(=O)OC1(CCN(C)CCCc2nc3ccccc3[nH]2)CCc2cc(F...,0.0,0,0.062469,0,0.025686,0,0.038081,0,0.014490,0,0.136364,0


In [51]:
df_toxic = df_preds[df_preds["Toxicity"] ==1.0]
df_toxic

,Name,SMILES,Toxicity,pred_rf,pred_rf_proba,pred_lgbm,pred_lgbm_proba,pred_svm,pred_svm_proba,pred_ridge,pred_ridge_proba,pred_knn,pred_knn_proba,mode
8,DILI_8,O=C(O)c1cccnc1,1.0,1,0.838148,1,0.821889,1,0.846727,1,0.835445,1,0.863636,1
11,DILI_11,NC(=O)c1cnccn1,1.0,1,0.629383,1,0.739691,1,0.673338,1,0.742622,1,0.681818,1
14,DILI_14,Cc1ccc2cc3c(ccc4ccccc43)c3c2c1CC3,1.0,1,0.791333,1,0.769701,1,0.547579,1,0.506498,1,0.954545,1
15,DILI_15,Oc1cccc2cccnc12,1.0,0,0.379259,1,0.630170,1,0.718572,1,0.708737,1,0.590909,1
17,DILI_17,CC(=O)Nc1nnc(S(N)(=O)=O)s1,1.0,0,0.201296,0,0.064791,0,0.221829,0,0.277418,0,0.136364,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
396,DILI_396,COC1C=COC2(C)Oc3c(C)c(O)c4c(c3C2=O)C2=NC3(CCN(...,1.0,0,0.161317,0,0.256121,0,0.169837,0,0.064415,0,0.318182,0
398,DILI_398,CN1CCC(Nc2ncc3ncnc(Nc4ccc(F)c(Cl)c4)c3n2)CC1,1.0,0,0.044321,0,0.050291,0,0.116721,0,0.100017,0,0.090909,0
399,DILI_399,Cc1ccc(Nc2nccc(N(C)c3ccc4c(C)n(C)nc4c3)n2)cc1S...,1.0,0,0.274691,0,0.090602,0,0.309576,0,0.331919,0,0.181818,0
401,DILI_401,C=C1c2cccc(O)c2C(O)=C2C(=O)C3(O)C(O)=C(C(N)=O)...,1.0,1,0.553531,1,0.526834,0,0.460178,0,0.484271,0,0.500000,0


In [52]:
df_pred_toxic_top = df_toxic[(df_toxic["pred_rf_proba"] >=0.8) & (df_toxic["pred_lgbm_proba"]  >=0.8) & (df_toxic["pred_svm_proba"]  >=0.8) & 
(df_toxic["pred_ridge_proba"]  >=0.8) & (df_toxic["pred_knn_proba"]  >=0.8)  ]
df_pred_toxic_top

,Name,SMILES,Toxicity,pred_rf,pred_rf_proba,pred_lgbm,pred_lgbm_proba,pred_svm,pred_svm_proba,pred_ridge,pred_ridge_proba,pred_knn,pred_knn_proba,mode
8,DILI_8,O=C(O)c1cccnc1,1.0,1,0.838148,1,0.821889,1,0.846727,1,0.835445,1,0.863636,1
232,DILI_232,S=C=Nc1cccc2ccccc12,1.0,1,0.962222,1,0.974709,1,0.902985,1,0.874835,1,1.000000,1


In [53]:
import mols2grid

In [54]:
mols2grid.display(df_pred_toxic_top,smiles_col="SMILES",subset=["img","Name","Toxicity", "pred_ridge_proba", ],selection=False, addStereoAnnotation=True, size=(300, 250) )

MolGridWidget()

In [55]:
df_pred_toxic_last = df_toxic[(df_toxic["pred_rf_proba"] <=0.2) & (df_toxic["pred_lgbm_proba"]  <=0.2) & (df_toxic["pred_svm_proba"]  <=0.2)
& (df_toxic["pred_ridge_proba"]  <=0.2) & (df_toxic["pred_ridge_proba"]  <=0.2) ]
df_pred_toxic_last

,Name,SMILES,Toxicity,pred_rf,pred_rf_proba,pred_lgbm,pred_lgbm_proba,pred_svm,pred_svm_proba,pred_ridge,pred_ridge_proba,pred_knn,pred_knn_proba,mode
46,DILI_46,CC1=C(C(=O)O)N2C(=O)C(NC(=O)C(N)c3ccc(O)cc3)C2SC1,1.0,0,0.017531,0,0.023837,0,0.099136,0,0.124697,0,0.090909,0
47,DILI_47,Cc1ccc(-c2cc(C(F)(F)F)nn2-c2ccc(S(N)(=O)=O)cc2...,1.0,0,0.189136,0,0.090931,0,0.184957,0,0.151015,0,0.090909,0
66,DILI_66,OCCN(CCO)c1nc(N2CCCCC2)c2nc(N(CCO)CCO)nc(N3CCC...,1.0,0,0.124840,0,0.102167,0,0.180460,0,0.089138,0,0.181818,0
73,DILI_73,CCc1cccc2c3c([nH]c12)C(CC)(CC(=O)O)OCC3,1.0,0,0.043704,0,0.125217,0,0.163287,0,0.152362,0,0.454545,0
106,DILI_106,CCn1cc(C(=O)O)c(=O)c2cc(F)c(N3CCNC(C)C3)c(F)c21,1.0,0,0.043316,0,0.024551,0,0.034774,0,0.017673,0,0.136364,0
115,DILI_115,Cc1ccccc1N1C(=O)c2cc(S(N)(=O)=O)c(Cl)cc2NC1C,1.0,0,0.102469,0,0.041497,0,0.146603,0,0.137140,0,0.272727,0
123,DILI_123,CCn1cc(C(=O)O)c(=O)c2ccc(C)nc21,1.0,0,0.182963,0,0.129499,0,0.150436,0,0.127083,0,0.272727,0
126,DILI_126,O=C(CCNNC(=O)c1ccncc1)NCc1ccccc1,1.0,0,0.194568,0,0.099698,0,0.147678,0,0.190315,0,0.454545,0
133,DILI_133,CCn1cc(C(=O)O)c(=O)c2cc(F)c(N3CCNCC3)cc21,1.0,0,0.140254,0,0.024182,0,0.047905,0,0.026889,0,0.136364,0
161,DILI_161,Cc1nc2n(c(=O)c1CCN1CCC(c3noc4cc(F)ccc34)CC1)CCCC2,1.0,0,0.007704,0,0.019864,0,0.045963,0,0.031833,0,0.000000,0


In [56]:
mols2grid.display(df_pred_toxic_last,smiles_col="SMILES",subset=["img","Name","Toxicity", "pred_ridge_proba"],selection=False, addStereoAnnotation=True, size=(300, 250))

MolGridWidget()

In [57]:
df_nontoxic= df_preds[df_preds["Toxicity"] == 0.0]
df_nontoxic

,Name,SMILES,Toxicity,pred_rf,pred_rf_proba,pred_lgbm,pred_lgbm_proba,pred_svm,pred_svm_proba,pred_ridge,pred_ridge_proba,pred_knn,pred_knn_proba,mode
0,DILI_0,CC(=O)OCC[N+](C)(C)C,0.0,1,0.836790,1,0.718751,1,0.666596,1,0.527748,0,0.454545,1
1,DILI_1,C[N+](C)(C)CC(=O)[O-],0.0,1,0.752593,1,0.861657,1,0.807273,1,0.710851,1,0.772727,1
2,DILI_2,O=C(NC(CO)C(O)c1ccc([N+](=O)[O-])cc1)C(Cl)Cl,0.0,0,0.454074,1,0.648162,1,0.760184,1,0.918027,0,0.500000,1
3,DILI_3,O=C(O)c1ccccc1O,0.0,1,0.832593,1,0.646671,1,0.795398,1,0.800162,1,0.636364,1
4,DILI_4,CC(NC(C)(C)C)C(=O)c1cccc(Cl)c1,0.0,1,0.722840,1,0.760705,1,0.699934,1,0.667100,1,0.818182,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
393,DILI_393,CC1CN(CC(Cc2ccccc2)C(=O)NCC(=O)O)CCC1(C)c1cccc...,0.0,0,0.443881,0,0.086836,0,0.049233,0,0.038025,0,0.136364,0
395,DILI_395,COCCCOc1cc(CC(CC(N)C(O)CC(C(=O)NCC(C)(C)C(N)=O...,0.0,0,0.044198,0,0.031931,0,0.106647,0,0.131582,0,0.136364,0
397,DILI_397,CC(=O)Oc1cc2c(s1)CCN(C(C(=O)C1CC1)c1ccccc1F)C2,0.0,0,0.081728,0,0.052572,0,0.066913,0,0.044240,0,0.181818,0
400,DILI_400,CC1C=CC=CCCC=CC=CC=CC=CC(OC2OC(C)C(O)C(N)C2O)C...,0.0,0,0.115062,0,0.184623,0,0.243432,0,0.325142,0,0.318182,0


In [58]:
df_nontoxic_top = df_nontoxic[(df_nontoxic["pred_rf_proba"] <=0.2) & (df_nontoxic["pred_lgbm_proba"]  <=0.2) & (df_nontoxic["pred_svm_proba"]  <=0.2)
& (df_nontoxic["pred_ridge_proba"]  <=0.2) & (df_nontoxic["pred_ridge_proba"]  <=0.2) ]
df_nontoxic_top

,Name,SMILES,Toxicity,pred_rf,pred_rf_proba,pred_lgbm,pred_lgbm_proba,pred_svm,pred_svm_proba,pred_ridge,pred_ridge_proba,pred_knn,pred_knn_proba,mode
21,DILI_21,CC(C)(C)NCC(O)c1ccc(O)c(CO)c1,0.0,0,0.036444,0,0.039581,0,0.151051,0,0.166072,0,0.000000,0
32,DILI_32,COc1ccc(CCN2CCC(Nc3nc4ccccc4n3Cc3ccc(F)cc3)CC2...,0.0,0,0.029160,0,0.032742,0,0.073926,0,0.053052,0,0.000000,0
33,DILI_33,CC(C)NCC(O)COc1ccc(CC(N)=O)cc1,0.0,0,0.088123,0,0.034099,0,0.153454,0,0.164261,0,0.090909,0
39,DILI_39,O=C1CC2(CCCC2)CC(=O)N1CCCCN1CCN(c2ncccn2)CC1,0.0,0,0.073333,0,0.072883,0,0.072710,0,0.051710,0,0.090909,0
43,DILI_43,CCN(CC)CCOCCOC(=O)C1(c2ccccc2)CCCC1,0.0,0,0.171457,0,0.104282,0,0.156006,0,0.095570,0,0.181818,0
44,DILI_44,CN(C)CCOC(c1ccc(Cl)cc1)c1ccccn1,0.0,0,0.061365,0,0.073974,0,0.165480,0,0.128729,0,0.136364,0
53,DILI_53,O=C1CCc2cc(OCCCCc3nnnn3C3CCCCC3)ccc2N1,0.0,0,0.079753,0,0.020495,0,0.179313,0,0.148139,0,0.045455,0
60,DILI_60,CNCCCN1c2ccccc2CCc2ccccc21,0.0,0,0.073827,0,0.047096,0,0.108189,0,0.093044,0,0.045455,0
68,DILI_68,COc1cc2c(cc1OC)C(=O)C(CC1CCN(Cc3ccccc3)CC1)C2,0.0,0,0.027654,0,0.057912,0,0.086834,0,0.043761,0,0.045455,0
69,DILI_69,CN(C)CCOC(C)(c1ccccc1)c1ccccn1,0.0,0,0.075185,0,0.040189,0,0.161910,0,0.143091,0,0.181818,0


In [59]:
mols2grid.display(df_nontoxic_top,smiles_col="SMILES",subset=["img","Name","Toxicity", "pred_ridge_proba", ],selection=False, addStereoAnnotation=True, size=(300, 250))

MolGridWidget()

In [60]:
df_nontoxic_last = df_nontoxic[(df_nontoxic["pred_rf_proba"] >=0.85) & (df_nontoxic["pred_lgbm_proba"]  >=0.85) & (df_nontoxic["pred_svm_proba"] >=0.85) 
& (df_nontoxic["pred_ridge_proba"] >=0.85) & (df_nontoxic["pred_knn_proba"]  >=0.85)]
df_nontoxic_last

,Name,SMILES,Toxicity,pred_rf,pred_rf_proba,pred_lgbm,pred_lgbm_proba,pred_svm,pred_svm_proba,pred_ridge,pred_ridge_proba,pred_knn,pred_knn_proba,mode
121,DILI_121,Clc1ccc(C(c2ccccc2Cl)C(Cl)Cl)cc1,0.0,1,0.988148,1,0.958109,1,0.864118,1,0.85705,1,0.909091,1


In [61]:
mols2grid.display(df_nontoxic_last,smiles_col="SMILES",subset=["img","Name","Toxicity", "pred_ridge_proba", ],selection=False, addStereoAnnotation=True, size=(300, 250))

MolGridWidget()

In [62]:
mat_met


,Metric,rf,lgbm,svm,ridge,knn,Combined Models
0,TP,40.000000,51.000000,49.000000,57.000000,39.000000,45.000000
1,TN,164.000000,168.000000,159.000000,159.000000,164.000000,165.000000
2,FP,47.000000,43.000000,52.000000,52.000000,47.000000,46.000000
3,FN,153.000000,142.000000,144.000000,136.000000,154.000000,148.000000
4,Accuracy,0.504950,0.542079,0.514851,0.534653,0.502475,0.519802
5,Precision,0.459770,0.542553,0.485149,0.522936,0.453488,0.494505
6,Sensitivity,0.207254,0.264249,0.253886,0.295337,0.202073,0.233161
7,Specificity,0.777300,0.796200,0.753600,0.753600,0.777300,0.782000
8,F1 score,0.285714,0.355401,0.333333,0.377483,0.279570,0.316901
9,F1 score (weighted),0.460937,0.506607,0.482362,0.508562,0.457389,0.480306


## Skin Reaction Set

In [3]:
df = pd.read_pickle('tdc_skin_reaction_FP.csv')
print(len(df))
df.head()

347


,Name,SMILES,fp_MACCS,Toxicity
0,SKIN_0,CC(=O)OCC(=O)C1(O)C(C)CC2C3CCC4=CC(=O)C=CC4(C)...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0.0
1,SKIN_1,CCCOc1ccc(Br)c(C(=O)c2ccc(OC)cc2O)c1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0.0
2,SKIN_2,CC=C(C)C=O,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0.0
3,SKIN_3,O=C1C([PH](c2ccccc2)(c2ccccc2)c2ccccc2)CCN1c1c...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0.0
4,SKIN_4,C=C(C=CCC(C)C)CC,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0.0


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 347 entries, 0 to 346
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Name      347 non-null    object 
 1   SMILES    347 non-null    object 
 2   fp_MACCS  347 non-null    object 
 3   Toxicity  347 non-null    float64
dtypes: float64(1), object(3)
memory usage: 11.0+ KB


In [5]:
print("Number of toxic molecules in external set:", len(df[df["Toxicity"] == 1.0]))
print("Number of nontoxic molecules in external set:", len(df[df["Toxicity"] == 0.0]))

Number of toxic molecules in external set: 234
Number of nontoxic molecules in external set: 113


#### Test Each Model One by One

In [6]:
#By using Morgan fingerprints with radius of 3 and 1025 bits
X_internal = np.array(list((df_internal['fp_MACCS']))).astype(float)
print(X_internal.shape)
y_internal=df_internal.Toxicity
print(y_internal.head())

(6403, 167)
0    0
1    0
2    0
3    0
4    0
Name: Toxicity, dtype: int64


In [7]:
#By using Morgan fingerprints with radius of 3 and 1025 bits
X = np.array(list((df['fp_MACCS']))).astype(float)
print(X.shape)
y_real=df.Toxicity
print(y_real.head())

(347, 167)
0    0.0
1    0.0
2    0.0
3    0.0
4    0.0
Name: Toxicity, dtype: float64


In [8]:
optimizedCV_clf_rf =  joblib.load("FinalModels/RF_optimized_full.joblib")
y_pred_rf = optimizedCV_clf_rf.predict(X)

#add to a dataframe
df_preds = df[["Name", "SMILES", "Toxicity"]]
df_preds["pred_rf"] = y_pred_rf
df_preds["pred_rf_proba"] = optimizedCV_clf_rf.predict_proba(X).tolist()
df_preds["pred_rf_proba"] = df_preds["pred_rf_proba"].str[1]

conf_matrix_rf = confusion_matrix(y_real, y_pred_rf) 
TP_rf = conf_matrix_rf[1][1]
TN_rf = conf_matrix_rf[0][0]
FP_rf = conf_matrix_rf[0][1] 
FN_rf = conf_matrix_rf[1][0]
Accuracy_rf = accuracy_score(y_real, y_pred_rf)
Precision_rf = precision_score(y_real, y_pred_rf)
Sensitivity_rf = recall_score(y_real, y_pred_rf)
Specificity_rf = round( TN_rf / (TN_rf+FP_rf),4 )
f1_scores_rf = f1_score(y_real, y_pred_rf)
f1_scores_W_rf = f1_score(y_real, y_pred_rf, average="weighted")
f1_scores_M_rf = f1_score(y_real, y_pred_rf, average="macro")
BA_scores_rf = balanced_accuracy_score(y_real, y_pred_rf)
MCC_rf= matthews_corrcoef(y_real, y_pred_rf)
NPV_rf= round( TN_rf / (TN_rf+FN_rf),4 )
ROC_AUC_rf = roc_auc_score(y_real, y_pred_rf)

mat_met = pd.DataFrame({'Metric':['TP','TN','FP','FN','Accuracy','Precision','Sensitivity','Specificity','F1 score','F1 score (weighted)',
                            'F1 score (macro)', 'Balanced Accuracy','MCC','NPV','ROC_AUC'],     
                        'rf':[np.mean(TP_rf), np.mean(TN_rf),
                                    np.mean(FP_rf), np.mean(FN_rf),
                                    np.mean(Accuracy_rf),
                                    np.mean(Precision_rf),
                                    np.mean(Sensitivity_rf),
                                    np.mean(Specificity_rf),
                                    np.mean(f1_scores_rf),
                                    np.mean(f1_scores_W_rf), 
                                    np.mean(f1_scores_M_rf), 
                                    np.mean(BA_scores_rf), 
                                    np.mean(MCC_rf),
                                    np.mean(NPV_rf),
                                    np.mean(ROC_AUC_rf)],
                   }) 
    
print(mat_met)

                 Metric          rf
0                    TP  194.000000
1                    TN   55.000000
2                    FP   58.000000
3                    FN   40.000000
4              Accuracy    0.717579
5             Precision    0.769841
6           Sensitivity    0.829060
7           Specificity    0.486700
8              F1 score    0.798354
9   F1 score (weighted)    0.710589
10     F1 score (macro)    0.663600
11    Balanced Accuracy    0.657893
12                  MCC    0.331877
13                  NPV    0.578900
14              ROC_AUC    0.657893


In [9]:
optimizedCV_clf_lgbm =  joblib.load("FinalModels/LGBM_optimized_full.joblib")
y_pred_lgbm = optimizedCV_clf_lgbm.predict(X)
df_preds["pred_lgbm"] = y_pred_lgbm
df_preds["pred_lgbm_proba"] = optimizedCV_clf_lgbm.predict_proba(X).tolist()
df_preds["pred_lgbm_proba"] = df_preds["pred_lgbm_proba"].str[1]

conf_matrix_lgbm = confusion_matrix(y_real, y_pred_lgbm) 
TP_lgbm = conf_matrix_lgbm[1][1]
TN_lgbm = conf_matrix_lgbm[0][0]
FP_lgbm = conf_matrix_lgbm[0][1] 
FN_lgbm = conf_matrix_lgbm[1][0]
Accuracy_lgbm = accuracy_score(y_real, y_pred_lgbm)
Precision_lgbm = precision_score(y_real, y_pred_lgbm)
Sensitivity_lgbm = recall_score(y_real, y_pred_lgbm)
Specificity_lgbm = round( TN_lgbm / (TN_lgbm+FP_lgbm),4 )
f1_scores_lgbm = f1_score(y_real, y_pred_lgbm)
f1_scores_W_lgbm = f1_score(y_real, y_pred_lgbm, average="weighted")
f1_scores_M_lgbm = f1_score(y_real, y_pred_lgbm, average="macro")
BA_scores_lgbm = balanced_accuracy_score(y_real, y_pred_lgbm)
MCC_lgbm= matthews_corrcoef(y_real, y_pred_lgbm)
NPV_lgbm= round( TN_lgbm / (TN_lgbm+FN_lgbm),4 )
ROC_AUC_lgbm = roc_auc_score(y_real, y_pred_lgbm)

lgbm = pd.DataFrame({  'lgbm':[np.mean(TP_lgbm), np.mean(TN_lgbm),
                                    np.mean(FP_lgbm), np.mean(FN_lgbm),
                                    np.mean(Accuracy_lgbm),
                                    np.mean(Precision_lgbm),
                                    np.mean(Sensitivity_lgbm),
                                    np.mean(Specificity_lgbm),
                                    np.mean(f1_scores_lgbm),
                                    np.mean(f1_scores_W_lgbm), 
                                    np.mean(f1_scores_M_lgbm), 
                                    np.mean(BA_scores_lgbm), 
                                    np.mean(MCC_lgbm),
                                    np.mean(NPV_lgbm),
                                    np.mean(ROC_AUC_lgbm)],
                 }) 

mat_met['lgbm'] = lgbm  
    
print(mat_met)

                 Metric          rf        lgbm
0                    TP  194.000000  202.000000
1                    TN   55.000000   47.000000
2                    FP   58.000000   66.000000
3                    FN   40.000000   32.000000
4              Accuracy    0.717579    0.717579
5             Precision    0.769841    0.753731
6           Sensitivity    0.829060    0.863248
7           Specificity    0.486700    0.415900
8              F1 score    0.798354    0.804781
9   F1 score (weighted)    0.710589    0.702137
10     F1 score (macro)    0.663600    0.647182
11    Balanced Accuracy    0.657893    0.639589
12                  MCC    0.331877    0.311994
13                  NPV    0.578900    0.594900
14              ROC_AUC    0.657893    0.639589


In [10]:
optimizedCV_clf_svm =  joblib.load("FinalModels/SVC_optimized_full.joblib")
y_pred_svm = optimizedCV_clf_svm.predict(X)
df_preds["pred_svm"] = y_pred_svm
df_preds["pred_svm_proba"] = optimizedCV_clf_svm.predict_proba(X).tolist()
df_preds["pred_svm_proba"] = df_preds["pred_svm_proba"].str[1]

conf_matrix_svm = confusion_matrix(y_real, y_pred_svm) 
TP_svm = conf_matrix_svm[1][1]
TN_svm = conf_matrix_svm[0][0]
FP_svm = conf_matrix_svm[0][1] 
FN_svm = conf_matrix_svm[1][0]
Accuracy_svm = accuracy_score(y_real, y_pred_svm)
Precision_svm = precision_score(y_real, y_pred_svm)
Sensitivity_svm = recall_score(y_real, y_pred_svm)
Specificity_svm = round( TN_svm / (TN_svm+FP_svm),4 )
f1_scores_svm = f1_score(y_real, y_pred_svm)
f1_scores_W_svm = f1_score(y_real, y_pred_svm, average="weighted")
f1_scores_M_svm = f1_score(y_real, y_pred_svm, average="macro")
BA_scores_svm = balanced_accuracy_score(y_real, y_pred_svm)
MCC_svm= matthews_corrcoef(y_real, y_pred_svm)
NPV_svm= round( TN_svm / (TN_svm+FN_svm),4 )
ROC_AUC_svm = roc_auc_score(y_real, y_pred_svm)

svm = pd.DataFrame({  'svm':[np.mean(TP_svm), np.mean(TN_svm),
                                    np.mean(FP_svm), np.mean(FN_svm),
                                    np.mean(Accuracy_svm),
                                    np.mean(Precision_svm),
                                    np.mean(Sensitivity_svm),
                                    np.mean(Specificity_svm),
                                    np.mean(f1_scores_svm),
                                    np.mean(f1_scores_W_svm), 
                                    np.mean(f1_scores_M_svm), 
                                    np.mean(BA_scores_svm), 
                                    np.mean(MCC_svm),
                                    np.mean(NPV_svm),
                                    np.mean(ROC_AUC_svm)],
                 }) 

mat_met['svm'] = svm  
print(mat_met)

                 Metric          rf        lgbm         svm
0                    TP  194.000000  202.000000  200.000000
1                    TN   55.000000   47.000000   47.000000
2                    FP   58.000000   66.000000   66.000000
3                    FN   40.000000   32.000000   34.000000
4              Accuracy    0.717579    0.717579    0.711816
5             Precision    0.769841    0.753731    0.751880
6           Sensitivity    0.829060    0.863248    0.854701
7           Specificity    0.486700    0.415900    0.415900
8              F1 score    0.798354    0.804781    0.800000
9   F1 score (weighted)    0.710589    0.702137    0.697270
10     F1 score (macro)    0.663600    0.647182    0.642268
11    Balanced Accuracy    0.657893    0.639589    0.635315
12                  MCC    0.331877    0.311994    0.299806
13                  NPV    0.578900    0.594900    0.580200
14              ROC_AUC    0.657893    0.639589    0.635315


In [11]:
optimizedCV_clf_ridge =  joblib.load("FinalModels/Ridge_optimized_full.joblib")

from sklearn.calibration import CalibratedClassifierCV
ridge_cal = CalibratedClassifierCV(
    optimizedCV_clf_ridge,
    method="sigmoid",   # Platt scaling
    cv="prefit"
)

ridge_cal.fit(X_internal, y_internal)

y_pred_ridge = ridge_cal.predict(X)
df_preds["pred_ridge"] = y_pred_ridge
df_preds["pred_ridge_proba"] = ridge_cal.predict_proba(X).tolist()
df_preds["pred_ridge_proba"] = df_preds["pred_ridge_proba"].str[1]
df_preds["pred_ridge"] = y_pred_ridge
#df_preds["pred_ridge_proba"] = optimizedCV_clf_ridge.decision_function(X).tolist()
#df_preds["pred_ridge_proba"] = df_preds["pred_ridge_proba"].str[1]

conf_matrix_ridge = confusion_matrix(y_real, y_pred_ridge) 
TP_ridge = conf_matrix_ridge[1][1]
TN_ridge = conf_matrix_ridge[0][0]
FP_ridge = conf_matrix_ridge[0][1] 
FN_ridge = conf_matrix_ridge[1][0]
Accuracy_ridge = accuracy_score(y_real, y_pred_ridge)
Precision_ridge = precision_score(y_real, y_pred_ridge)
Sensitivity_ridge = recall_score(y_real, y_pred_ridge)
Specificity_ridge = round( TN_ridge / (TN_ridge+FP_ridge),4 )
f1_scores_ridge = f1_score(y_real, y_pred_ridge)
f1_scores_W_ridge = f1_score(y_real, y_pred_ridge, average="weighted")
f1_scores_M_ridge = f1_score(y_real, y_pred_ridge, average="macro")
BA_scores_ridge = balanced_accuracy_score(y_real, y_pred_ridge)
MCC_ridge= matthews_corrcoef(y_real, y_pred_ridge)
NPV_ridge= round( TN_ridge / (TN_ridge+FN_ridge),4 )
ROC_AUC_ridge = ROC_AUC_ridge = roc_auc_score(y_real,
    optimizedCV_clf_ridge.decision_function(X))

ridge = pd.DataFrame({  'ridge':[np.mean(TP_ridge), np.mean(TN_ridge),
                                    np.mean(FP_ridge), np.mean(FN_ridge),
                                    np.mean(Accuracy_ridge),
                                    np.mean(Precision_ridge),
                                    np.mean(Sensitivity_ridge),
                                    np.mean(Specificity_ridge),
                                    np.mean(f1_scores_ridge),
                                    np.mean(f1_scores_W_ridge), 
                                    np.mean(f1_scores_M_ridge), 
                                    np.mean(BA_scores_ridge), 
                                    np.mean(MCC_ridge),
                                    np.mean(NPV_ridge),
                                    np.mean(ROC_AUC_ridge)],
                 }) 

mat_met['ridge'] = ridge  
print(mat_met)

                 Metric          rf        lgbm         svm       ridge
0                    TP  194.000000  202.000000  200.000000  208.000000
1                    TN   55.000000   47.000000   47.000000   49.000000
2                    FP   58.000000   66.000000   66.000000   64.000000
3                    FN   40.000000   32.000000   34.000000   26.000000
4              Accuracy    0.717579    0.717579    0.711816    0.740634
5             Precision    0.769841    0.753731    0.751880    0.764706
6           Sensitivity    0.829060    0.863248    0.854701    0.888889
7           Specificity    0.486700    0.415900    0.415900    0.433600
8              F1 score    0.798354    0.804781    0.800000    0.822134
9   F1 score (weighted)    0.710589    0.702137    0.697270    0.724161
10     F1 score (macro)    0.663600    0.647182    0.642268    0.671705
11    Balanced Accuracy    0.657893    0.639589    0.635315    0.661259
12                  MCC    0.331877    0.311994    0.299806    0

In [12]:
optimizedCV_clf_knn =  joblib.load("FinalModels/KNN_optimized_full.joblib")
y_pred_knn = optimizedCV_clf_knn.predict(X)
df_preds["pred_knn"] = y_pred_knn
df_preds["pred_knn_proba"] = optimizedCV_clf_knn.predict_proba(X).tolist()
df_preds["pred_knn_proba"] = df_preds["pred_knn_proba"].str[1]

conf_matrix_knn = confusion_matrix(y_real, y_pred_knn) 
TP_knn = conf_matrix_knn[1][1]
TN_knn = conf_matrix_knn[0][0]
FP_knn = conf_matrix_knn[0][1] 
FN_knn = conf_matrix_knn[1][0]
Accuracy_knn = accuracy_score(y_real, y_pred_knn)
Precision_knn = precision_score(y_real, y_pred_knn)
Sensitivity_knn = recall_score(y_real, y_pred_knn)
Specificity_knn = round( TN_knn / (TN_knn+FP_knn),4 )
f1_scores_knn = f1_score(y_real, y_pred_knn)
f1_scores_W_knn = f1_score(y_real, y_pred_knn, average="weighted")
f1_scores_M_knn = f1_score(y_real, y_pred_knn, average="macro")
BA_scores_knn = balanced_accuracy_score(y_real, y_pred_knn)
MCC_knn= matthews_corrcoef(y_real, y_pred_knn)
NPV_knn= round( TN_knn / (TN_knn+FN_knn),4 )
ROC_AUC_knn = roc_auc_score(y_real, y_pred_knn)

knn = pd.DataFrame({  'knn':[np.mean(TP_knn), np.mean(TN_knn),
                                    np.mean(FP_knn), np.mean(FN_knn),
                                    np.mean(Accuracy_knn),
                                    np.mean(Precision_knn),
                                    np.mean(Sensitivity_knn),
                                    np.mean(Specificity_knn),
                                    np.mean(f1_scores_knn),
                                    np.mean(f1_scores_W_knn), 
                                    np.mean(f1_scores_M_knn), 
                                    np.mean(BA_scores_knn), 
                                    np.mean(MCC_knn),
                                    np.mean(NPV_knn),
                                    np.mean(ROC_AUC_knn)],
                 }) 

mat_met['knn'] = knn  
print(mat_met)

                 Metric          rf        lgbm         svm       ridge  \
0                    TP  194.000000  202.000000  200.000000  208.000000   
1                    TN   55.000000   47.000000   47.000000   49.000000   
2                    FP   58.000000   66.000000   66.000000   64.000000   
3                    FN   40.000000   32.000000   34.000000   26.000000   
4              Accuracy    0.717579    0.717579    0.711816    0.740634   
5             Precision    0.769841    0.753731    0.751880    0.764706   
6           Sensitivity    0.829060    0.863248    0.854701    0.888889   
7           Specificity    0.486700    0.415900    0.415900    0.433600   
8              F1 score    0.798354    0.804781    0.800000    0.822134   
9   F1 score (weighted)    0.710589    0.702137    0.697270    0.724161   
10     F1 score (macro)    0.663600    0.647182    0.642268    0.671705   
11    Balanced Accuracy    0.657893    0.639589    0.635315    0.661259   
12                  MCC  

In [13]:
df_preds

,Name,SMILES,Toxicity,pred_rf,pred_rf_proba,pred_lgbm,pred_lgbm_proba,pred_svm,pred_svm_proba,pred_ridge,pred_ridge_proba,pred_knn,pred_knn_proba
0,SKIN_0,CC(=O)OCC(=O)C1(O)C(C)CC2C3CCC4=CC(=O)C=CC4(C)...,0.0,0,0.081596,0,0.141323,0,0.121558,0,0.271813,0,0.142857
1,SKIN_1,CCCOc1ccc(Br)c(C(=O)c2ccc(OC)cc2O)c1,0.0,0,0.403509,0,0.484538,0,0.380849,0,0.480206,1,0.571429
2,SKIN_2,CC=C(C)C=O,0.0,1,1.000000,1,0.990141,1,0.965538,1,0.966566,1,1.000000
3,SKIN_3,O=C1C([PH](c2ccccc2)(c2ccccc2)c2ccccc2)CCN1c1c...,0.0,0,0.216374,0,0.145619,0,0.236731,0,0.437087,0,0.428571
4,SKIN_4,C=C(C=CCC(C)C)CC,0.0,1,0.947368,1,0.972814,1,0.916931,1,0.971641,1,1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
342,SKIN_342,Nc1ccc(N)c2c1C(=O)c1ccccc1C2=O,1.0,1,0.863158,1,0.863836,1,0.846962,1,0.529005,1,1.000000
343,SKIN_343,Nc1ccc(N(CO)CCO)cc1[N+](=O)[O-],1.0,0,0.473684,1,0.893136,1,0.928538,1,0.866427,1,0.714286
344,SKIN_344,O=C1C=CC(=O)O1,1.0,1,0.925146,1,0.921438,1,0.867041,1,0.932276,1,0.857143
345,SKIN_345,CCCOC(=O)c1cc(O)c(O)c(O)c1,1.0,1,0.693442,1,0.550253,1,0.732516,1,0.575637,1,0.571429


In [14]:
df_preds["mode"] = df_preds[["pred_rf", "pred_lgbm", "pred_svm","pred_ridge", "pred_knn" ]].mode(axis=1)
df_preds

,Name,SMILES,Toxicity,pred_rf,pred_rf_proba,pred_lgbm,pred_lgbm_proba,pred_svm,pred_svm_proba,pred_ridge,pred_ridge_proba,pred_knn,pred_knn_proba,mode
0,SKIN_0,CC(=O)OCC(=O)C1(O)C(C)CC2C3CCC4=CC(=O)C=CC4(C)...,0.0,0,0.081596,0,0.141323,0,0.121558,0,0.271813,0,0.142857,0
1,SKIN_1,CCCOc1ccc(Br)c(C(=O)c2ccc(OC)cc2O)c1,0.0,0,0.403509,0,0.484538,0,0.380849,0,0.480206,1,0.571429,0
2,SKIN_2,CC=C(C)C=O,0.0,1,1.000000,1,0.990141,1,0.965538,1,0.966566,1,1.000000,1
3,SKIN_3,O=C1C([PH](c2ccccc2)(c2ccccc2)c2ccccc2)CCN1c1c...,0.0,0,0.216374,0,0.145619,0,0.236731,0,0.437087,0,0.428571,0
4,SKIN_4,C=C(C=CCC(C)C)CC,0.0,1,0.947368,1,0.972814,1,0.916931,1,0.971641,1,1.000000,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
342,SKIN_342,Nc1ccc(N)c2c1C(=O)c1ccccc1C2=O,1.0,1,0.863158,1,0.863836,1,0.846962,1,0.529005,1,1.000000,1
343,SKIN_343,Nc1ccc(N(CO)CCO)cc1[N+](=O)[O-],1.0,0,0.473684,1,0.893136,1,0.928538,1,0.866427,1,0.714286,1
344,SKIN_344,O=C1C=CC(=O)O1,1.0,1,0.925146,1,0.921438,1,0.867041,1,0.932276,1,0.857143,1
345,SKIN_345,CCCOC(=O)c1cc(O)c(O)c(O)c1,1.0,1,0.693442,1,0.550253,1,0.732516,1,0.575637,1,0.571429,1


In [15]:
y_pred_ave = df_preds["mode"]

conf_matrix_ave = confusion_matrix(y_real, y_pred_ave) 
TP_ave = conf_matrix_ave[1][1]
TN_ave = conf_matrix_ave[0][0]
FP_ave = conf_matrix_ave[0][1] 
FN_ave = conf_matrix_ave[1][0]
Accuracy_ave = accuracy_score(y_real, y_pred_ave)
Precision_ave = precision_score(y_real, y_pred_ave)
Sensitivity_ave = recall_score(y_real, y_pred_ave)
Specificity_ave = round( TN_ave / (TN_ave+FP_ave),4 )
f1_scores_ave = f1_score(y_real, y_pred_ave)
f1_scores_W_ave = f1_score(y_real, y_pred_ave, average="weighted")
f1_scores_M_ave = f1_score(y_real, y_pred_ave, average="macro")
BA_scores_ave = balanced_accuracy_score(y_real, y_pred_ave)
MCC_ave= matthews_corrcoef(y_real, y_pred_ave)
NPV_ave= round( TN_ave / (TN_ave+FN_ave),4 )
ROC_AUC_ave = roc_auc_score(y_real, y_pred_ave)

ave = pd.DataFrame({  'Combined Models':[np.mean(TP_ave), np.mean(TN_ave),
                                    np.mean(FP_ave), np.mean(FN_ave),
                                    np.mean(Accuracy_ave),
                                    np.mean(Precision_ave),
                                    np.mean(Sensitivity_ave),
                                    np.mean(Specificity_ave),
                                    np.mean(f1_scores_ave),
                                    np.mean(f1_scores_W_ave), 
                                    np.mean(f1_scores_M_ave), 
                                    np.mean(BA_scores_ave), 
                                    np.mean(MCC_ave),
                                    np.mean(NPV_ave),
                                    np.mean(ROC_AUC_ave)],
                 }) 

mat_met['Combined Models'] = ave  
print(mat_met)

                 Metric          rf        lgbm         svm       ridge  \
0                    TP  194.000000  202.000000  200.000000  208.000000   
1                    TN   55.000000   47.000000   47.000000   49.000000   
2                    FP   58.000000   66.000000   66.000000   64.000000   
3                    FN   40.000000   32.000000   34.000000   26.000000   
4              Accuracy    0.717579    0.717579    0.711816    0.740634   
5             Precision    0.769841    0.753731    0.751880    0.764706   
6           Sensitivity    0.829060    0.863248    0.854701    0.888889   
7           Specificity    0.486700    0.415900    0.415900    0.433600   
8              F1 score    0.798354    0.804781    0.800000    0.822134   
9   F1 score (weighted)    0.710589    0.702137    0.697270    0.724161   
10     F1 score (macro)    0.663600    0.647182    0.642268    0.671705   
11    Balanced Accuracy    0.657893    0.639589    0.635315    0.661259   
12                  MCC  

In [16]:
#df_preds.to_csv("input/External_FTO_IC50_singleProtein_RelationEqual_Predictions.csv", index=False)

In [17]:
#mat_met.to_csv("Statistics_ExternalSet_RelationEqual.csv",index=False)

In [18]:
mat_met.to_csv("Evaluation_tdc_skin.csv",index=False)

In [19]:
df_preds.sort_values(["pred_ridge_proba"], ascending=False)

,Name,SMILES,Toxicity,pred_rf,pred_rf_proba,pred_lgbm,pred_lgbm_proba,pred_svm,pred_svm_proba,pred_ridge,pred_ridge_proba,pred_knn,pred_knn_proba,mode
236,SKIN_236,O=[N+]([O-])c1ccc(S(=O)(=O)O)c([N+](=O)[O-])c1,1.0,1,0.849123,1,0.995818,1,0.947823,1,0.972226,1,1.000000,1
4,SKIN_4,C=C(C=CCC(C)C)CC,0.0,1,0.947368,1,0.972814,1,0.916931,1,0.971641,1,1.000000,1
190,SKIN_190,C=C(C)C(=O)O,1.0,1,0.978947,1,0.973884,1,0.918235,1,0.970494,1,1.000000,1
2,SKIN_2,CC=C(C)C=O,0.0,1,1.000000,1,0.990141,1,0.965538,1,0.966566,1,1.000000,1
328,SKIN_328,CN(C)C(=S)SSC(=S)N(C)C,1.0,1,0.954386,1,0.992374,1,0.936511,1,0.966278,1,0.857143,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46,SKIN_46,CCCCN1C(=O)C(C(O)C2CCCCC2)NC(=O)C12CCN(Cc1ccc(...,0.0,0,0.092398,0,0.143610,0,0.102903,0,0.053823,0,0.142857,0
33,SKIN_33,COc1cc(C)c2c(Oc3cccc(C(F)(F)F)c3)c(OC)cc(NC(C)...,0.0,0,0.078947,0,0.074589,0,0.189498,0,0.045246,0,0.142857,0
8,SKIN_8,CC(C)(CC1Cc2ccccc2C1)NCC(O)COc1cc(CCC(=O)O)cc(...,0.0,0,0.025146,0,0.012057,0,0.054786,0,0.022946,0,0.000000,0
310,SKIN_310,CCN(CC)CCN(Cc1ccc(-c2ccc(C(F)(F)F)cc2)cc1)C(=O...,1.0,0,0.061404,0,0.028763,0,0.123055,0,0.022849,0,0.000000,0


In [20]:
df_toxic = df_preds[df_preds["Toxicity"] ==1.0]
df_toxic

,Name,SMILES,Toxicity,pred_rf,pred_rf_proba,pred_lgbm,pred_lgbm_proba,pred_svm,pred_svm_proba,pred_ridge,pred_ridge_proba,pred_knn,pred_knn_proba,mode
109,SKIN_109,CCCCCCCC=CC=O,1.0,1,0.998830,1,0.933981,1,0.894846,1,0.854204,1,1.000000,1
110,SKIN_110,CC=Cc1ccc(OC)cc1,1.0,1,0.954386,1,0.950191,1,0.858520,1,0.882750,1,0.857143,1
111,SKIN_111,CCCCCC(=O)CC(=O)c1ccccc1,1.0,1,0.817836,1,0.760418,1,0.780578,1,0.503556,1,1.000000,1
112,SKIN_112,CCc1ccc(CC)c(C(=O)CC(C)=O)c1,1.0,1,0.696199,1,0.852141,1,0.826918,1,0.642199,1,0.857143,1
113,SKIN_113,CC(=O)CC(=O)c1cc(C)ccc1C,1.0,1,0.702632,1,0.789968,1,0.768602,1,0.582466,1,0.857143,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
342,SKIN_342,Nc1ccc(N)c2c1C(=O)c1ccccc1C2=O,1.0,1,0.863158,1,0.863836,1,0.846962,1,0.529005,1,1.000000,1
343,SKIN_343,Nc1ccc(N(CO)CCO)cc1[N+](=O)[O-],1.0,0,0.473684,1,0.893136,1,0.928538,1,0.866427,1,0.714286,1
344,SKIN_344,O=C1C=CC(=O)O1,1.0,1,0.925146,1,0.921438,1,0.867041,1,0.932276,1,0.857143,1
345,SKIN_345,CCCOC(=O)c1cc(O)c(O)c(O)c1,1.0,1,0.693442,1,0.550253,1,0.732516,1,0.575637,1,0.571429,1


In [21]:
df_pred_toxic_top = df_toxic[(df_toxic["pred_rf_proba"] >=0.8) & (df_toxic["pred_lgbm_proba"]  >=0.8) & (df_toxic["pred_svm_proba"]  >=0.8) & 
(df_toxic["pred_ridge_proba"]  >=0.8) & (df_toxic["pred_knn_proba"]  >=0.8)  ]
df_pred_toxic_top

,Name,SMILES,Toxicity,pred_rf,pred_rf_proba,pred_lgbm,pred_lgbm_proba,pred_svm,pred_svm_proba,pred_ridge,pred_ridge_proba,pred_knn,pred_knn_proba,mode
109,SKIN_109,CCCCCCCC=CC=O,1.0,1,0.998830,1,0.933981,1,0.894846,1,0.854204,1,1.000000,1
110,SKIN_110,CC=Cc1ccc(OC)cc1,1.0,1,0.954386,1,0.950191,1,0.858520,1,0.882750,1,0.857143,1
114,SKIN_114,N#CCCC(Br)(C#N)CBr,1.0,1,0.971930,1,0.974436,1,0.806820,1,0.932628,1,1.000000,1
119,SKIN_119,O=[N+]([O-])c1ccc(Cl)c([N+](=O)[O-])c1,1.0,1,0.806550,1,0.994377,1,0.895933,1,0.955158,1,1.000000,1
120,SKIN_120,CCCCCCCCCCCCCCCCCCCl,1.0,1,0.982456,1,0.971701,1,0.932322,1,0.929126,1,1.000000,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
324,SKIN_324,CCOC(=S)S,1.0,1,0.919298,1,0.877651,1,0.834800,1,0.926628,1,0.857143,1
328,SKIN_328,CN(C)C(=S)SSC(=S)N(C)C,1.0,1,0.954386,1,0.992374,1,0.936511,1,0.966278,1,0.857143,1
330,SKIN_330,C=CCCCCCCCCC=O,1.0,1,0.947368,1,0.988214,1,0.938372,1,0.937464,1,1.000000,1
335,SKIN_335,O=C1OC(=O)C2CCCCC12,1.0,1,0.894503,1,0.932395,1,0.918800,1,0.883120,1,1.000000,1


In [22]:
import mols2grid

In [23]:
mols2grid.display(df_pred_toxic_top,smiles_col="SMILES",subset=["img","Name","Toxicity", "pred_ridge_proba", ],selection=False, addStereoAnnotation=True, size=(300, 250) )

MolGridWidget()

In [24]:
df_pred_toxic_last = df_toxic[(df_toxic["pred_rf_proba"] <=0.2) & (df_toxic["pred_lgbm_proba"]  <=0.2) & (df_toxic["pred_svm_proba"]  <=0.2)
& (df_toxic["pred_ridge_proba"]  <=0.2) & (df_toxic["pred_ridge_proba"]  <=0.2) ]
df_pred_toxic_last

,Name,SMILES,Toxicity,pred_rf,pred_rf_proba,pred_lgbm,pred_lgbm_proba,pred_svm,pred_svm_proba,pred_ridge,pred_ridge_proba,pred_knn,pred_knn_proba,mode
126,SKIN_126,CN(C)CCOC(C)(c1ccccc1)c1ccc(Br)cc1,1.0,0,0.075439,0,0.066048,0,0.126285,0,0.156287,0,0.000000,0
127,SKIN_127,CC(C)(C)N(CC(=O)c1ccc(O)c(CO)c1)Cc1ccccc1,1.0,0,0.133918,0,0.058915,0,0.104401,0,0.115386,0,0.000000,0
256,SKIN_256,CCN(CCNS(C)(=O)=O)c1ccc(N)c(C)c1,1.0,0,0.081170,0,0.023794,0,0.060933,0,0.115231,0,0.142857,0
276,SKIN_276,CCCCCn1c(=O)[nH]c(=O)c2[nH]c(Cl)nc21,1.0,0,0.130526,0,0.118742,0,0.187358,0,0.130448,0,0.000000,0
310,SKIN_310,CCN(CC)CCN(Cc1ccc(-c2ccc(C(F)(F)F)cc2)cc1)C(=O...,1.0,0,0.061404,0,0.028763,0,0.123055,0,0.022849,0,0.000000,0
315,SKIN_315,CC1(C)SC2C(NC(=O)Cc3ccccc3)C(=O)N2C1C(=O)O,1.0,0,0.079657,0,0.125405,0,0.163100,0,0.127591,0,0.142857,0


In [25]:
mols2grid.display(df_pred_toxic_last,smiles_col="SMILES",subset=["img","Name","Toxicity", "pred_ridge_proba"],selection=False, addStereoAnnotation=True, size=(300, 250))

MolGridWidget()

In [26]:
df_nontoxic= df_preds[df_preds["Toxicity"] == 0.0]
df_nontoxic

,Name,SMILES,Toxicity,pred_rf,pred_rf_proba,pred_lgbm,pred_lgbm_proba,pred_svm,pred_svm_proba,pred_ridge,pred_ridge_proba,pred_knn,pred_knn_proba,mode
0,SKIN_0,CC(=O)OCC(=O)C1(O)C(C)CC2C3CCC4=CC(=O)C=CC4(C)...,0.0,0,0.081596,0,0.141323,0,0.121558,0,0.271813,0,0.142857,0
1,SKIN_1,CCCOc1ccc(Br)c(C(=O)c2ccc(OC)cc2O)c1,0.0,0,0.403509,0,0.484538,0,0.380849,0,0.480206,1,0.571429,0
2,SKIN_2,CC=C(C)C=O,0.0,1,1.000000,1,0.990141,1,0.965538,1,0.966566,1,1.000000,1
3,SKIN_3,O=C1C([PH](c2ccccc2)(c2ccccc2)c2ccccc2)CCN1c1c...,0.0,0,0.216374,0,0.145619,0,0.236731,0,0.437087,0,0.428571,0
4,SKIN_4,C=C(C=CCC(C)C)CC,0.0,1,0.947368,1,0.972814,1,0.916931,1,0.971641,1,1.000000,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
108,SKIN_108,COc1cc(C=O)ccc1O,0.0,1,0.914620,1,0.832474,1,0.895723,1,0.786876,1,0.857143,1
331,SKIN_331,C[N+](C)(C)c1ccc2c(c1)C(=NNc1ccc(N)c([N+](=O)[...,0.0,1,0.518596,1,0.946133,1,0.621835,0,0.487578,1,0.714286,1
332,SKIN_332,CCOC(=O)C(C)(C)Oc1ccc(Cl)cc1,0.0,1,0.597368,1,0.773160,1,0.842226,1,0.672700,1,0.571429,1
333,SKIN_333,CCCCCCCCC(=O)O,0.0,0,0.487527,1,0.553289,1,0.690773,1,0.763680,0,0.428571,1


In [27]:
df_nontoxic_top = df_nontoxic[(df_nontoxic["pred_rf_proba"] <=0.2) & (df_nontoxic["pred_lgbm_proba"]  <=0.2) & (df_nontoxic["pred_svm_proba"]  <=0.2)
& (df_nontoxic["pred_ridge_proba"]  <=0.2) & (df_nontoxic["pred_ridge_proba"]  <=0.2) ]
df_nontoxic_top

,Name,SMILES,Toxicity,pred_rf,pred_rf_proba,pred_lgbm,pred_lgbm_proba,pred_svm,pred_svm_proba,pred_ridge,pred_ridge_proba,pred_knn,pred_knn_proba,mode
8,SKIN_8,CC(C)(CC1Cc2ccccc2C1)NCC(O)COc1cc(CCC(=O)O)cc(...,0.0,0,0.025146,0,0.012057,0,0.054786,0,0.022946,0,0.000000,0
19,SKIN_19,N#CC1CC(F)CN1C(=O)C(N)C(c1ccc(F)cc1)c1ccc(F)cc1,0.0,0,0.068421,0,0.057958,0,0.116423,0,0.192459,0,0.142857,0
27,SKIN_27,CC(C)(C)N(Cc1ccccc1)CC(O)c1ccc(O)c(CO)c1,0.0,0,0.139766,0,0.059003,0,0.106301,0,0.093605,0,0.000000,0
33,SKIN_33,COc1cc(C)c2c(Oc3cccc(C(F)(F)F)c3)c(OC)cc(NC(C)...,0.0,0,0.078947,0,0.074589,0,0.189498,0,0.045246,0,0.142857,0
39,SKIN_39,CCC(C)C1C(=O)NC(C2Cc3ccccc3C2)C(=O)N1C(C(=O)N1...,0.0,0,0.037251,0,0.144632,0,0.174192,0,0.144781,0,0.285714,0
41,SKIN_41,CC(C)N(C(=O)CN1C(=O)C(N)C(=O)N(c2ccccc2)c2cccc...,0.0,0,0.105848,0,0.012023,0,0.071971,0,0.073510,0,0.142857,0
46,SKIN_46,CCCCN1C(=O)C(C(O)C2CCCCC2)NC(=O)C12CCN(Cc1ccc(...,0.0,0,0.092398,0,0.143610,0,0.102903,0,0.053823,0,0.142857,0
62,SKIN_62,COc1ccc(Nc2ccc(CCNCC(O)c3ccc(O)c4[nH]c(=O)ccc3...,0.0,0,0.045029,0,0.044630,0,0.057333,0,0.064330,0,0.000000,0


In [28]:
mols2grid.display(df_nontoxic_top,smiles_col="SMILES",subset=["img","Name","Toxicity", "pred_ridge_proba", ],selection=False, addStereoAnnotation=True, size=(300, 250))

MolGridWidget()

In [29]:
df_nontoxic_last = df_nontoxic[(df_nontoxic["pred_rf_proba"] >=0.85) & (df_nontoxic["pred_lgbm_proba"]  >=0.85) & (df_nontoxic["pred_svm_proba"] >=0.85) 
& (df_nontoxic["pred_ridge_proba"] >=0.85) & (df_nontoxic["pred_knn_proba"]  >=0.85)]
df_nontoxic_last

,Name,SMILES,Toxicity,pred_rf,pred_rf_proba,pred_lgbm,pred_lgbm_proba,pred_svm,pred_svm_proba,pred_ridge,pred_ridge_proba,pred_knn,pred_knn_proba,mode
2,SKIN_2,CC=C(C)C=O,0.0,1,1.000000,1,0.990141,1,0.965538,1,0.966566,1,1.000000,1
4,SKIN_4,C=C(C=CCC(C)C)CC,0.0,1,0.947368,1,0.972814,1,0.916931,1,0.971641,1,1.000000,1
43,SKIN_43,CC1C=CC(C(C)C)CC1,0.0,1,0.968421,1,0.888379,1,0.873670,1,0.869873,1,0.857143,1
51,SKIN_51,C=C1CCC(C(C)C)CC1,0.0,1,0.968421,1,0.956361,1,0.873696,1,0.934412,1,1.000000,1
65,SKIN_65,CCCCBr,0.0,1,0.996491,1,0.988228,1,0.959700,1,0.945780,1,1.000000,1
66,SKIN_66,CCCCCCCCCBr,0.0,1,0.978947,1,0.963811,1,0.920931,1,0.928349,1,1.000000,1
67,SKIN_67,C=C1CC=C(C(C)C)CC1,0.0,1,0.968421,1,0.956361,1,0.873696,1,0.934412,1,1.000000,1
69,SKIN_69,CCCCCCCCCCl,0.0,1,0.982456,1,0.971701,1,0.932322,1,0.929126,1,1.000000,1
79,SKIN_79,O=C(O)C=CC(=O)O,0.0,1,0.978363,1,0.910001,1,0.887335,1,0.945613,1,1.000000,1
82,SKIN_82,CCCCCC,0.0,1,0.989474,1,0.957835,1,0.899526,1,0.910620,1,0.857143,1


In [30]:
mols2grid.display(df_nontoxic_last,smiles_col="SMILES",subset=["img","Name","Toxicity", "pred_ridge_proba", ],selection=False, addStereoAnnotation=True, size=(300, 250))

MolGridWidget()

In [31]:
mat_met


,Metric,rf,lgbm,svm,ridge,knn,Combined Models
0,TP,194.000000,202.000000,200.000000,208.000000,194.000000,202.000000
1,TN,55.000000,47.000000,47.000000,49.000000,40.000000,49.000000
2,FP,58.000000,66.000000,66.000000,64.000000,73.000000,64.000000
3,FN,40.000000,32.000000,34.000000,26.000000,40.000000,32.000000
4,Accuracy,0.717579,0.717579,0.711816,0.740634,0.674352,0.723343
5,Precision,0.769841,0.753731,0.751880,0.764706,0.726592,0.759398
6,Sensitivity,0.829060,0.863248,0.854701,0.888889,0.829060,0.863248
7,Specificity,0.486700,0.415900,0.415900,0.433600,0.354000,0.433600
8,F1 score,0.798354,0.804781,0.800000,0.822134,0.774451,0.808000
9,F1 score (weighted),0.710589,0.702137,0.697270,0.724161,0.657236,0.709379


## Cancer Set

In [3]:
df = pd.read_pickle('tdc_carcinogens_lagunin_FP.csv')
print(len(df))
df.head()

236


,Name,SMILES,fp_MACCS,Toxicity
0,CAN_0,CC(CCl)OC(C)CCl,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1.0
1,CAN_1,COc1ccc(C(=O)/C(Br)=C\C(=O)O)cc1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1.0
2,CAN_2,O=S(=O)(O)c1cc(S(=O)(=O)O)c2c(/N=N/c3ccccc3)c(...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1.0
3,CAN_3,O=S(=O)(O)c1ccc(/N=N/c2cc(S(=O)(=O)O)c3ccccc3c...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1.0
4,CAN_4,Nc1ccc(/N=N/c2ccc(-c3ccc(/N=N/c4c(S(=O)(=O)O)c...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1.0


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 236 entries, 0 to 235
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Name      236 non-null    object 
 1   SMILES    236 non-null    object 
 2   fp_MACCS  236 non-null    object 
 3   Toxicity  236 non-null    float64
dtypes: float64(1), object(3)
memory usage: 7.5+ KB


In [5]:
print("Number of toxic molecules in external set:", len(df[df["Toxicity"] == 1.0]))
print("Number of nontoxic molecules in external set:", len(df[df["Toxicity"] == 0.0]))

Number of toxic molecules in external set: 48
Number of nontoxic molecules in external set: 188


#### Test Each Model One by One

In [6]:
#By using Morgan fingerprints with radius of 3 and 1025 bits
X_internal = np.array(list((df_internal['fp_MACCS']))).astype(float)
print(X_internal.shape)
y_internal=df_internal.Toxicity
print(y_internal.head())

(6403, 167)
0    0
1    0
2    0
3    0
4    0
Name: Toxicity, dtype: int64


In [7]:
#By using Morgan fingerprints with radius of 3 and 1025 bits
X = np.array(list((df['fp_MACCS']))).astype(float)
print(X.shape)
y_real=df.Toxicity
print(y_real.head())

(236, 167)
0    1.0
1    1.0
2    1.0
3    1.0
4    1.0
Name: Toxicity, dtype: float64


In [8]:
optimizedCV_clf_rf =  joblib.load("FinalModels/RF_optimized_full.joblib")
y_pred_rf = optimizedCV_clf_rf.predict(X)

#add to a dataframe
df_preds = df[["Name", "SMILES", "Toxicity"]]
df_preds["pred_rf"] = y_pred_rf
df_preds["pred_rf_proba"] = optimizedCV_clf_rf.predict_proba(X).tolist()
df_preds["pred_rf_proba"] = df_preds["pred_rf_proba"].str[1]

conf_matrix_rf = confusion_matrix(y_real, y_pred_rf) 
TP_rf = conf_matrix_rf[1][1]
TN_rf = conf_matrix_rf[0][0]
FP_rf = conf_matrix_rf[0][1] 
FN_rf = conf_matrix_rf[1][0]
Accuracy_rf = accuracy_score(y_real, y_pred_rf)
Precision_rf = precision_score(y_real, y_pred_rf)
Sensitivity_rf = recall_score(y_real, y_pred_rf)
Specificity_rf = round( TN_rf / (TN_rf+FP_rf),4 )
f1_scores_rf = f1_score(y_real, y_pred_rf)
f1_scores_W_rf = f1_score(y_real, y_pred_rf, average="weighted")
f1_scores_M_rf = f1_score(y_real, y_pred_rf, average="macro")
BA_scores_rf = balanced_accuracy_score(y_real, y_pred_rf)
MCC_rf= matthews_corrcoef(y_real, y_pred_rf)
NPV_rf= round( TN_rf / (TN_rf+FN_rf),4 )
ROC_AUC_rf = roc_auc_score(y_real, y_pred_rf)

mat_met = pd.DataFrame({'Metric':['TP','TN','FP','FN','Accuracy','Precision','Sensitivity','Specificity','F1 score','F1 score (weighted)',
                            'F1 score (macro)', 'Balanced Accuracy','MCC','NPV','ROC_AUC'],     
                        'rf':[np.mean(TP_rf), np.mean(TN_rf),
                                    np.mean(FP_rf), np.mean(FN_rf),
                                    np.mean(Accuracy_rf),
                                    np.mean(Precision_rf),
                                    np.mean(Sensitivity_rf),
                                    np.mean(Specificity_rf),
                                    np.mean(f1_scores_rf),
                                    np.mean(f1_scores_W_rf), 
                                    np.mean(f1_scores_M_rf), 
                                    np.mean(BA_scores_rf), 
                                    np.mean(MCC_rf),
                                    np.mean(NPV_rf),
                                    np.mean(ROC_AUC_rf)],
                   }) 
    
print(mat_met)

                 Metric          rf
0                    TP   31.000000
1                    TN  125.000000
2                    FP   63.000000
3                    FN   17.000000
4              Accuracy    0.661017
5             Precision    0.329787
6           Sensitivity    0.645833
7           Specificity    0.664900
8              F1 score    0.436620
9   F1 score (weighted)    0.692297
10     F1 score (macro)    0.597098
11    Balanced Accuracy    0.655363
12                  MCC    0.255488
13                  NPV    0.880300
14              ROC_AUC    0.655363


In [9]:
optimizedCV_clf_lgbm =  joblib.load("FinalModels/LGBM_optimized_full.joblib")
y_pred_lgbm = optimizedCV_clf_lgbm.predict(X)
df_preds["pred_lgbm"] = y_pred_lgbm
df_preds["pred_lgbm_proba"] = optimizedCV_clf_lgbm.predict_proba(X).tolist()
df_preds["pred_lgbm_proba"] = df_preds["pred_lgbm_proba"].str[1]

conf_matrix_lgbm = confusion_matrix(y_real, y_pred_lgbm) 
TP_lgbm = conf_matrix_lgbm[1][1]
TN_lgbm = conf_matrix_lgbm[0][0]
FP_lgbm = conf_matrix_lgbm[0][1] 
FN_lgbm = conf_matrix_lgbm[1][0]
Accuracy_lgbm = accuracy_score(y_real, y_pred_lgbm)
Precision_lgbm = precision_score(y_real, y_pred_lgbm)
Sensitivity_lgbm = recall_score(y_real, y_pred_lgbm)
Specificity_lgbm = round( TN_lgbm / (TN_lgbm+FP_lgbm),4 )
f1_scores_lgbm = f1_score(y_real, y_pred_lgbm)
f1_scores_W_lgbm = f1_score(y_real, y_pred_lgbm, average="weighted")
f1_scores_M_lgbm = f1_score(y_real, y_pred_lgbm, average="macro")
BA_scores_lgbm = balanced_accuracy_score(y_real, y_pred_lgbm)
MCC_lgbm= matthews_corrcoef(y_real, y_pred_lgbm)
NPV_lgbm= round( TN_lgbm / (TN_lgbm+FN_lgbm),4 )
ROC_AUC_lgbm = roc_auc_score(y_real, y_pred_lgbm)

lgbm = pd.DataFrame({  'lgbm':[np.mean(TP_lgbm), np.mean(TN_lgbm),
                                    np.mean(FP_lgbm), np.mean(FN_lgbm),
                                    np.mean(Accuracy_lgbm),
                                    np.mean(Precision_lgbm),
                                    np.mean(Sensitivity_lgbm),
                                    np.mean(Specificity_lgbm),
                                    np.mean(f1_scores_lgbm),
                                    np.mean(f1_scores_W_lgbm), 
                                    np.mean(f1_scores_M_lgbm), 
                                    np.mean(BA_scores_lgbm), 
                                    np.mean(MCC_lgbm),
                                    np.mean(NPV_lgbm),
                                    np.mean(ROC_AUC_lgbm)],
                 }) 

mat_met['lgbm'] = lgbm  
    
print(mat_met)

                 Metric          rf        lgbm
0                    TP   31.000000   38.000000
1                    TN  125.000000  127.000000
2                    FP   63.000000   61.000000
3                    FN   17.000000   10.000000
4              Accuracy    0.661017    0.699153
5             Precision    0.329787    0.383838
6           Sensitivity    0.645833    0.791667
7           Specificity    0.664900    0.675500
8              F1 score    0.436620    0.517007
9   F1 score (weighted)    0.692297    0.727735
10     F1 score (macro)    0.597098    0.649273
11    Balanced Accuracy    0.655363    0.733599
12                  MCC    0.255488    0.381086
13                  NPV    0.880300    0.927000
14              ROC_AUC    0.655363    0.733599


In [10]:
optimizedCV_clf_svm =  joblib.load("FinalModels/SVC_optimized_full.joblib")
y_pred_svm = optimizedCV_clf_svm.predict(X)
df_preds["pred_svm"] = y_pred_svm
df_preds["pred_svm_proba"] = optimizedCV_clf_svm.predict_proba(X).tolist()
df_preds["pred_svm_proba"] = df_preds["pred_svm_proba"].str[1]

conf_matrix_svm = confusion_matrix(y_real, y_pred_svm) 
TP_svm = conf_matrix_svm[1][1]
TN_svm = conf_matrix_svm[0][0]
FP_svm = conf_matrix_svm[0][1] 
FN_svm = conf_matrix_svm[1][0]
Accuracy_svm = accuracy_score(y_real, y_pred_svm)
Precision_svm = precision_score(y_real, y_pred_svm)
Sensitivity_svm = recall_score(y_real, y_pred_svm)
Specificity_svm = round( TN_svm / (TN_svm+FP_svm),4 )
f1_scores_svm = f1_score(y_real, y_pred_svm)
f1_scores_W_svm = f1_score(y_real, y_pred_svm, average="weighted")
f1_scores_M_svm = f1_score(y_real, y_pred_svm, average="macro")
BA_scores_svm = balanced_accuracy_score(y_real, y_pred_svm)
MCC_svm= matthews_corrcoef(y_real, y_pred_svm)
NPV_svm= round( TN_svm / (TN_svm+FN_svm),4 )
ROC_AUC_svm = roc_auc_score(y_real, y_pred_svm)

svm = pd.DataFrame({  'svm':[np.mean(TP_svm), np.mean(TN_svm),
                                    np.mean(FP_svm), np.mean(FN_svm),
                                    np.mean(Accuracy_svm),
                                    np.mean(Precision_svm),
                                    np.mean(Sensitivity_svm),
                                    np.mean(Specificity_svm),
                                    np.mean(f1_scores_svm),
                                    np.mean(f1_scores_W_svm), 
                                    np.mean(f1_scores_M_svm), 
                                    np.mean(BA_scores_svm), 
                                    np.mean(MCC_svm),
                                    np.mean(NPV_svm),
                                    np.mean(ROC_AUC_svm)],
                 }) 

mat_met['svm'] = svm  
print(mat_met)

                 Metric          rf        lgbm         svm
0                    TP   31.000000   38.000000   32.000000
1                    TN  125.000000  127.000000  134.000000
2                    FP   63.000000   61.000000   54.000000
3                    FN   17.000000   10.000000   16.000000
4              Accuracy    0.661017    0.699153    0.703390
5             Precision    0.329787    0.383838    0.372093
6           Sensitivity    0.645833    0.791667    0.666667
7           Specificity    0.664900    0.675500    0.712800
8              F1 score    0.436620    0.517007    0.477612
9   F1 score (weighted)    0.692297    0.727735    0.728773
10     F1 score (macro)    0.597098    0.649273    0.635256
11    Balanced Accuracy    0.655363    0.733599    0.689716
12                  MCC    0.255488    0.381086    0.317351
13                  NPV    0.880300    0.927000    0.893300
14              ROC_AUC    0.655363    0.733599    0.689716


In [11]:
optimizedCV_clf_ridge =  joblib.load("FinalModels/Ridge_optimized_full.joblib")

from sklearn.calibration import CalibratedClassifierCV
ridge_cal = CalibratedClassifierCV(
    optimizedCV_clf_ridge,
    method="sigmoid",   # Platt scaling
    cv="prefit"
)

ridge_cal.fit(X_internal, y_internal)

y_pred_ridge = ridge_cal.predict(X)
df_preds["pred_ridge"] = y_pred_ridge
df_preds["pred_ridge_proba"] = ridge_cal.predict_proba(X).tolist()
df_preds["pred_ridge_proba"] = df_preds["pred_ridge_proba"].str[1]
df_preds["pred_ridge"] = y_pred_ridge
#df_preds["pred_ridge_proba"] = optimizedCV_clf_ridge.decision_function(X).tolist()
#df_preds["pred_ridge_proba"] = df_preds["pred_ridge_proba"].str[1]

conf_matrix_ridge = confusion_matrix(y_real, y_pred_ridge) 
TP_ridge = conf_matrix_ridge[1][1]
TN_ridge = conf_matrix_ridge[0][0]
FP_ridge = conf_matrix_ridge[0][1] 
FN_ridge = conf_matrix_ridge[1][0]
Accuracy_ridge = accuracy_score(y_real, y_pred_ridge)
Precision_ridge = precision_score(y_real, y_pred_ridge)
Sensitivity_ridge = recall_score(y_real, y_pred_ridge)
Specificity_ridge = round( TN_ridge / (TN_ridge+FP_ridge),4 )
f1_scores_ridge = f1_score(y_real, y_pred_ridge)
f1_scores_W_ridge = f1_score(y_real, y_pred_ridge, average="weighted")
f1_scores_M_ridge = f1_score(y_real, y_pred_ridge, average="macro")
BA_scores_ridge = balanced_accuracy_score(y_real, y_pred_ridge)
MCC_ridge= matthews_corrcoef(y_real, y_pred_ridge)
NPV_ridge= round( TN_ridge / (TN_ridge+FN_ridge),4 )
ROC_AUC_ridge = ROC_AUC_ridge = roc_auc_score(y_real,
    optimizedCV_clf_ridge.decision_function(X))

ridge = pd.DataFrame({  'ridge':[np.mean(TP_ridge), np.mean(TN_ridge),
                                    np.mean(FP_ridge), np.mean(FN_ridge),
                                    np.mean(Accuracy_ridge),
                                    np.mean(Precision_ridge),
                                    np.mean(Sensitivity_ridge),
                                    np.mean(Specificity_ridge),
                                    np.mean(f1_scores_ridge),
                                    np.mean(f1_scores_W_ridge), 
                                    np.mean(f1_scores_M_ridge), 
                                    np.mean(BA_scores_ridge), 
                                    np.mean(MCC_ridge),
                                    np.mean(NPV_ridge),
                                    np.mean(ROC_AUC_ridge)],
                 }) 

mat_met['ridge'] = ridge  
print(mat_met)

                 Metric          rf        lgbm         svm       ridge
0                    TP   31.000000   38.000000   32.000000   35.000000
1                    TN  125.000000  127.000000  134.000000  130.000000
2                    FP   63.000000   61.000000   54.000000   58.000000
3                    FN   17.000000   10.000000   16.000000   13.000000
4              Accuracy    0.661017    0.699153    0.703390    0.699153
5             Precision    0.329787    0.383838    0.372093    0.376344
6           Sensitivity    0.645833    0.791667    0.666667    0.729167
7           Specificity    0.664900    0.675500    0.712800    0.691500
8              F1 score    0.436620    0.517007    0.477612    0.496454
9   F1 score (weighted)    0.692297    0.727735    0.728773    0.726710
10     F1 score (macro)    0.597098    0.649273    0.635256    0.640976
11    Balanced Accuracy    0.655363    0.733599    0.689716    0.710328
12                  MCC    0.255488    0.381086    0.317351    0

In [12]:
optimizedCV_clf_knn =  joblib.load("FinalModels/KNN_optimized_full.joblib")
y_pred_knn = optimizedCV_clf_knn.predict(X)
df_preds["pred_knn"] = y_pred_knn
df_preds["pred_knn_proba"] = optimizedCV_clf_knn.predict_proba(X).tolist()
df_preds["pred_knn_proba"] = df_preds["pred_knn_proba"].str[1]

conf_matrix_knn = confusion_matrix(y_real, y_pred_knn) 
TP_knn = conf_matrix_knn[1][1]
TN_knn = conf_matrix_knn[0][0]
FP_knn = conf_matrix_knn[0][1] 
FN_knn = conf_matrix_knn[1][0]
Accuracy_knn = accuracy_score(y_real, y_pred_knn)
Precision_knn = precision_score(y_real, y_pred_knn)
Sensitivity_knn = recall_score(y_real, y_pred_knn)
Specificity_knn = round( TN_knn / (TN_knn+FP_knn),4 )
f1_scores_knn = f1_score(y_real, y_pred_knn)
f1_scores_W_knn = f1_score(y_real, y_pred_knn, average="weighted")
f1_scores_M_knn = f1_score(y_real, y_pred_knn, average="macro")
BA_scores_knn = balanced_accuracy_score(y_real, y_pred_knn)
MCC_knn= matthews_corrcoef(y_real, y_pred_knn)
NPV_knn= round( TN_knn / (TN_knn+FN_knn),4 )
ROC_AUC_knn = roc_auc_score(y_real, y_pred_knn)

knn = pd.DataFrame({  'knn':[np.mean(TP_knn), np.mean(TN_knn),
                                    np.mean(FP_knn), np.mean(FN_knn),
                                    np.mean(Accuracy_knn),
                                    np.mean(Precision_knn),
                                    np.mean(Sensitivity_knn),
                                    np.mean(Specificity_knn),
                                    np.mean(f1_scores_knn),
                                    np.mean(f1_scores_W_knn), 
                                    np.mean(f1_scores_M_knn), 
                                    np.mean(BA_scores_knn), 
                                    np.mean(MCC_knn),
                                    np.mean(NPV_knn),
                                    np.mean(ROC_AUC_knn)],
                 }) 

mat_met['knn'] = knn  
print(mat_met)

                 Metric          rf        lgbm         svm       ridge  \
0                    TP   31.000000   38.000000   32.000000   35.000000   
1                    TN  125.000000  127.000000  134.000000  130.000000   
2                    FP   63.000000   61.000000   54.000000   58.000000   
3                    FN   17.000000   10.000000   16.000000   13.000000   
4              Accuracy    0.661017    0.699153    0.703390    0.699153   
5             Precision    0.329787    0.383838    0.372093    0.376344   
6           Sensitivity    0.645833    0.791667    0.666667    0.729167   
7           Specificity    0.664900    0.675500    0.712800    0.691500   
8              F1 score    0.436620    0.517007    0.477612    0.496454   
9   F1 score (weighted)    0.692297    0.727735    0.728773    0.726710   
10     F1 score (macro)    0.597098    0.649273    0.635256    0.640976   
11    Balanced Accuracy    0.655363    0.733599    0.689716    0.710328   
12                  MCC  

In [13]:
df_preds

,Name,SMILES,Toxicity,pred_rf,pred_rf_proba,pred_lgbm,pred_lgbm_proba,pred_svm,pred_svm_proba,pred_ridge,pred_ridge_proba,pred_knn,pred_knn_proba
0,CAN_0,CC(CCl)OC(C)CCl,1.0,1,0.980239,1,0.968557,1,0.930283,1,0.901434,1,1.000000
1,CAN_1,COc1ccc(C(=O)/C(Br)=C\C(=O)O)cc1,1.0,1,0.625714,1,0.742569,1,0.660487,1,0.778179,1,0.750000
2,CAN_2,O=S(=O)(O)c1cc(S(=O)(=O)O)c2c(/N=N/c3ccccc3)c(...,1.0,1,1.000000,1,0.980161,1,0.911417,1,0.869674,1,0.916667
3,CAN_3,O=S(=O)(O)c1ccc(/N=N/c2cc(S(=O)(=O)O)c3ccccc3c...,1.0,1,1.000000,1,0.980161,1,0.911417,1,0.869674,1,0.916667
4,CAN_4,Nc1ccc(/N=N/c2ccc(-c3ccc(/N=N/c4c(S(=O)(=O)O)c...,1.0,1,0.647576,1,0.915535,1,0.825217,1,0.718083,1,0.916667
...,...,...,...,...,...,...,...,...,...,...,...,...,...
231,CAN_231,COc1ccc(C[C@@](C)(N)C(=O)O)cc1OC,0.0,0,0.213106,0,0.151301,0,0.099695,0,0.288627,0,0.166667
232,CAN_232,CN1CCC(=C2c3ccccc3CCc3sccc32)CC1,0.0,0,0.072707,0,0.032488,0,0.106286,0,0.113880,0,0.083333
233,CAN_233,O=C(O)[C@]1(O)C[C@H](O)[C@@H](O)[C@@H](O)C1,0.0,1,0.707122,1,0.593930,1,0.824894,1,0.809364,1,0.750000
234,CAN_234,CCC(C)N1CCC2(CC1)N=C1C(=C3NC(=O)/C(C)=C/C=C/[C...,0.0,0,0.230783,0,0.342558,0,0.182650,0,0.060686,0,0.333333


In [14]:
df_preds["mode"] = df_preds[["pred_rf", "pred_lgbm", "pred_svm","pred_ridge", "pred_knn" ]].mode(axis=1)
df_preds

,Name,SMILES,Toxicity,pred_rf,pred_rf_proba,pred_lgbm,pred_lgbm_proba,pred_svm,pred_svm_proba,pred_ridge,pred_ridge_proba,pred_knn,pred_knn_proba,mode
0,CAN_0,CC(CCl)OC(C)CCl,1.0,1,0.980239,1,0.968557,1,0.930283,1,0.901434,1,1.000000,1
1,CAN_1,COc1ccc(C(=O)/C(Br)=C\C(=O)O)cc1,1.0,1,0.625714,1,0.742569,1,0.660487,1,0.778179,1,0.750000,1
2,CAN_2,O=S(=O)(O)c1cc(S(=O)(=O)O)c2c(/N=N/c3ccccc3)c(...,1.0,1,1.000000,1,0.980161,1,0.911417,1,0.869674,1,0.916667,1
3,CAN_3,O=S(=O)(O)c1ccc(/N=N/c2cc(S(=O)(=O)O)c3ccccc3c...,1.0,1,1.000000,1,0.980161,1,0.911417,1,0.869674,1,0.916667,1
4,CAN_4,Nc1ccc(/N=N/c2ccc(-c3ccc(/N=N/c4c(S(=O)(=O)O)c...,1.0,1,0.647576,1,0.915535,1,0.825217,1,0.718083,1,0.916667,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
231,CAN_231,COc1ccc(C[C@@](C)(N)C(=O)O)cc1OC,0.0,0,0.213106,0,0.151301,0,0.099695,0,0.288627,0,0.166667,0
232,CAN_232,CN1CCC(=C2c3ccccc3CCc3sccc32)CC1,0.0,0,0.072707,0,0.032488,0,0.106286,0,0.113880,0,0.083333,0
233,CAN_233,O=C(O)[C@]1(O)C[C@H](O)[C@@H](O)[C@@H](O)C1,0.0,1,0.707122,1,0.593930,1,0.824894,1,0.809364,1,0.750000,1
234,CAN_234,CCC(C)N1CCC2(CC1)N=C1C(=C3NC(=O)/C(C)=C/C=C/[C...,0.0,0,0.230783,0,0.342558,0,0.182650,0,0.060686,0,0.333333,0


In [15]:
y_pred_ave = df_preds["mode"]

conf_matrix_ave = confusion_matrix(y_real, y_pred_ave) 
TP_ave = conf_matrix_ave[1][1]
TN_ave = conf_matrix_ave[0][0]
FP_ave = conf_matrix_ave[0][1] 
FN_ave = conf_matrix_ave[1][0]
Accuracy_ave = accuracy_score(y_real, y_pred_ave)
Precision_ave = precision_score(y_real, y_pred_ave)
Sensitivity_ave = recall_score(y_real, y_pred_ave)
Specificity_ave = round( TN_ave / (TN_ave+FP_ave),4 )
f1_scores_ave = f1_score(y_real, y_pred_ave)
f1_scores_W_ave = f1_score(y_real, y_pred_ave, average="weighted")
f1_scores_M_ave = f1_score(y_real, y_pred_ave, average="macro")
BA_scores_ave = balanced_accuracy_score(y_real, y_pred_ave)
MCC_ave= matthews_corrcoef(y_real, y_pred_ave)
NPV_ave= round( TN_ave / (TN_ave+FN_ave),4 )
ROC_AUC_ave = roc_auc_score(y_real, y_pred_ave)

ave = pd.DataFrame({  'Combined Models':[np.mean(TP_ave), np.mean(TN_ave),
                                    np.mean(FP_ave), np.mean(FN_ave),
                                    np.mean(Accuracy_ave),
                                    np.mean(Precision_ave),
                                    np.mean(Sensitivity_ave),
                                    np.mean(Specificity_ave),
                                    np.mean(f1_scores_ave),
                                    np.mean(f1_scores_W_ave), 
                                    np.mean(f1_scores_M_ave), 
                                    np.mean(BA_scores_ave), 
                                    np.mean(MCC_ave),
                                    np.mean(NPV_ave),
                                    np.mean(ROC_AUC_ave)],
                 }) 

mat_met['Combined Models'] = ave  
print(mat_met)

                 Metric          rf        lgbm         svm       ridge  \
0                    TP   31.000000   38.000000   32.000000   35.000000   
1                    TN  125.000000  127.000000  134.000000  130.000000   
2                    FP   63.000000   61.000000   54.000000   58.000000   
3                    FN   17.000000   10.000000   16.000000   13.000000   
4              Accuracy    0.661017    0.699153    0.703390    0.699153   
5             Precision    0.329787    0.383838    0.372093    0.376344   
6           Sensitivity    0.645833    0.791667    0.666667    0.729167   
7           Specificity    0.664900    0.675500    0.712800    0.691500   
8              F1 score    0.436620    0.517007    0.477612    0.496454   
9   F1 score (weighted)    0.692297    0.727735    0.728773    0.726710   
10     F1 score (macro)    0.597098    0.649273    0.635256    0.640976   
11    Balanced Accuracy    0.655363    0.733599    0.689716    0.710328   
12                  MCC  

In [16]:
#df_preds.to_csv("input/External_FTO_IC50_singleProtein_RelationEqual_Predictions.csv", index=False)

In [17]:
#mat_met.to_csv("Statistics_ExternalSet_RelationEqual.csv",index=False)

In [18]:
mat_met.to_csv("Evaluation_tdc_cancer.csv",index=False)

In [19]:
df_preds.sort_values(["pred_rf_proba"], ascending=False)

,Name,SMILES,Toxicity,pred_rf,pred_rf_proba,pred_lgbm,pred_lgbm_proba,pred_svm,pred_svm_proba,pred_ridge,pred_ridge_proba,pred_knn,pred_knn_proba,mode
3,CAN_3,O=S(=O)(O)c1ccc(/N=N/c2cc(S(=O)(=O)O)c3ccccc3c...,1.0,1,1.000000,1,0.980161,1,0.911417,1,0.869674,1,0.916667,1
2,CAN_2,O=S(=O)(O)c1cc(S(=O)(=O)O)c2c(/N=N/c3ccccc3)c(...,1.0,1,1.000000,1,0.980161,1,0.911417,1,0.869674,1,0.916667,1
44,CAN_44,O=S(=O)(O)c1cc(S(=O)(=O)O)c2c(/N=N\c3ccc(S(=O)...,1.0,1,1.000000,1,0.980161,1,0.911417,1,0.869674,1,0.916667,1
7,CAN_7,O=C/C=C\O,1.0,1,0.990119,1,0.975884,1,0.966269,1,0.966451,1,1.000000,1
13,CAN_13,Cc1cc(C)c(/N=N/c2cc(S(=O)(=O)O)c3ccccc3c2O)c(S...,1.0,1,0.983221,1,0.955530,1,0.887455,1,0.803183,1,0.916667,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
162,CAN_162,COC(=O)[C@@H]1[C@@H](O)CC[C@H]2CN3CCc4c([nH]c5...,0.0,0,0.025447,0,0.218417,0,0.147738,0,0.060564,0,0.333333,0
54,CAN_54,CC1(C)[C@@H](OC(=O)CCC(=O)O)CC[C@@]2(C)[C@H]1C...,0.0,0,0.023490,0,0.181374,0,0.123033,0,0.317782,0,0.000000,0
163,CAN_163,COc1cc(N)c(Cl)cc1C(=O)NC1CCN(Cc2ccccc2)CC1,0.0,0,0.021160,0,0.054891,0,0.113586,0,0.069589,0,0.000000,0
158,CAN_158,O=C(c1ccc(F)cc1)C1CCN(CCn2c(=O)[nH]c3ccccc3c2=...,0.0,0,0.011745,0,0.033853,0,0.081191,0,0.040941,0,0.000000,0


In [20]:
df_toxic = df_preds[df_preds["Toxicity"] ==1.0]
df_toxic

,Name,SMILES,Toxicity,pred_rf,pred_rf_proba,pred_lgbm,pred_lgbm_proba,pred_svm,pred_svm_proba,pred_ridge,pred_ridge_proba,pred_knn,pred_knn_proba,mode
0,CAN_0,CC(CCl)OC(C)CCl,1.0,1,0.980239,1,0.968557,1,0.930283,1,0.901434,1,1.000000,1
1,CAN_1,COc1ccc(C(=O)/C(Br)=C\C(=O)O)cc1,1.0,1,0.625714,1,0.742569,1,0.660487,1,0.778179,1,0.750000,1
2,CAN_2,O=S(=O)(O)c1cc(S(=O)(=O)O)c2c(/N=N/c3ccccc3)c(...,1.0,1,1.000000,1,0.980161,1,0.911417,1,0.869674,1,0.916667,1
3,CAN_3,O=S(=O)(O)c1ccc(/N=N/c2cc(S(=O)(=O)O)c3ccccc3c...,1.0,1,1.000000,1,0.980161,1,0.911417,1,0.869674,1,0.916667,1
4,CAN_4,Nc1ccc(/N=N/c2ccc(-c3ccc(/N=N/c4c(S(=O)(=O)O)c...,1.0,1,0.647576,1,0.915535,1,0.825217,1,0.718083,1,0.916667,1
5,CAN_5,Nc1cc(S(=O)(=O)O)cc2cc(S(=O)(=O)O)c(/N=N/c3ccc...,1.0,1,0.643848,1,0.952008,1,0.853173,1,0.745933,1,0.916667,1
6,CAN_6,CC1(C)SC2C(NC(=O)[C@H](N)c3ccccc3)C(=O)N2[C@@H...,1.0,0,0.034638,0,0.008918,0,0.107610,0,0.167101,0,0.083333,0
7,CAN_7,O=C/C=C\O,1.0,1,0.990119,1,0.975884,1,0.966269,1,0.966451,1,1.000000,1
8,CAN_8,C=CCNN,1.0,1,0.872092,1,0.977189,1,0.918100,1,0.859778,1,0.916667,1
9,CAN_9,CCC1NC(=O)C2C(Cl)C(Cl)CN2C(=O)C(C(C)O)NC(=O)CC...,1.0,1,0.688367,1,0.586099,1,0.555029,0,0.157189,0,0.333333,1


In [21]:
df_pred_toxic_top = df_toxic[(df_toxic["pred_rf_proba"] >=0.8) & (df_toxic["pred_lgbm_proba"]  >=0.8) & (df_toxic["pred_svm_proba"]  >=0.8) & 
(df_toxic["pred_ridge_proba"]  >=0.8) & (df_toxic["pred_knn_proba"]  >=0.8)  ]
df_pred_toxic_top

,Name,SMILES,Toxicity,pred_rf,pred_rf_proba,pred_lgbm,pred_lgbm_proba,pred_svm,pred_svm_proba,pred_ridge,pred_ridge_proba,pred_knn,pred_knn_proba,mode
0,CAN_0,CC(CCl)OC(C)CCl,1.0,1,0.980239,1,0.968557,1,0.930283,1,0.901434,1,1.000000,1
2,CAN_2,O=S(=O)(O)c1cc(S(=O)(=O)O)c2c(/N=N/c3ccccc3)c(...,1.0,1,1.000000,1,0.980161,1,0.911417,1,0.869674,1,0.916667,1
3,CAN_3,O=S(=O)(O)c1ccc(/N=N/c2cc(S(=O)(=O)O)c3ccccc3c...,1.0,1,1.000000,1,0.980161,1,0.911417,1,0.869674,1,0.916667,1
7,CAN_7,O=C/C=C\O,1.0,1,0.990119,1,0.975884,1,0.966269,1,0.966451,1,1.000000,1
8,CAN_8,C=CCNN,1.0,1,0.872092,1,0.977189,1,0.918100,1,0.859778,1,0.916667,1
13,CAN_13,Cc1cc(C)c(/N=N/c2cc(S(=O)(=O)O)c3ccccc3c2O)c(S...,1.0,1,0.983221,1,0.955530,1,0.887455,1,0.803183,1,0.916667,1
14,CAN_14,O=S(=O)(O)c1ccc2c(/N=N/c3ccc(S(=O)(=O)O)c4cccc...,1.0,1,0.941834,1,0.966467,1,0.879354,1,0.861700,1,0.916667,1
33,CAN_33,C=CCNNCC=C,1.0,1,0.902442,1,0.953485,1,0.896163,1,0.856827,1,0.833333,1
44,CAN_44,O=S(=O)(O)c1cc(S(=O)(=O)O)c2c(/N=N\c3ccc(S(=O)...,1.0,1,1.000000,1,0.980161,1,0.911417,1,0.869674,1,0.916667,1


In [22]:
import mols2grid

In [23]:
mols2grid.display(df_pred_toxic_top,smiles_col="SMILES",subset=["img","Name","Toxicity", "pred_ridge_proba", ],selection=False, addStereoAnnotation=True, size=(300, 250) )

MolGridWidget()

In [24]:
df_pred_toxic_last = df_toxic[(df_toxic["pred_rf_proba"] <=0.2) & (df_toxic["pred_lgbm_proba"]  <=0.2) & (df_toxic["pred_svm_proba"]  <=0.2)
& (df_toxic["pred_ridge_proba"]  <=0.2) & (df_toxic["pred_ridge_proba"]  <=0.2) ]
df_pred_toxic_last

,Name,SMILES,Toxicity,pred_rf,pred_rf_proba,pred_lgbm,pred_lgbm_proba,pred_svm,pred_svm_proba,pred_ridge,pred_ridge_proba,pred_knn,pred_knn_proba,mode
6,CAN_6,CC1(C)SC2C(NC(=O)[C@H](N)c3ccccc3)C(=O)N2[C@@H...,1.0,0,0.034638,0,0.008918,0,0.107610,0,0.167101,0,0.083333,0
43,CAN_43,Cc1ccccc1OCC(O)CNCCn1cc(C)c(=O)[nH]c1=O,1.0,0,0.028635,0,0.022871,0,0.108415,0,0.032044,0,0.166667,0


In [25]:
mols2grid.display(df_pred_toxic_last,smiles_col="SMILES",subset=["img","Name","Toxicity", "pred_ridge_proba"],selection=False, addStereoAnnotation=True, size=(300, 250))

MolGridWidget()

In [26]:
df_nontoxic= df_preds[df_preds["Toxicity"] == 0.0]
df_nontoxic

,Name,SMILES,Toxicity,pred_rf,pred_rf_proba,pred_lgbm,pred_lgbm_proba,pred_svm,pred_svm_proba,pred_ridge,pred_ridge_proba,pred_knn,pred_knn_proba,mode
48,CAN_48,CC(=O)Nc1cc2ccccc2oc1=O,0.0,0,0.364840,0,0.271833,0,0.199195,0,0.355433,0,0.333333,0
49,CAN_49,CCN1C[C@@]2(COC)C3[C@@H](OC)[C@H]4[C@@H]1[C@@]...,0.0,0,0.491387,0,0.313801,0,0.228186,0,0.026092,0,0.250000,0
50,CAN_50,COC(=O)C1=CO[C@@H](C)[C@H]2CN3CCc4c([nH]c5cccc...,0.0,0,0.144389,0,0.369061,0,0.186667,0,0.158497,0,0.166667,0
51,CAN_51,CC[C@H](CC[C@@H](C)[C@H]1CC[C@H]2[C@@H]3CC=C4C...,0.0,1,0.698919,0,0.247372,0,0.208768,0,0.417775,0,0.416667,0
52,CAN_52,CO[C@@]1(NC(=O)Cc2cccs2)C(=O)N2C(C(=O)O)=C(COC...,0.0,0,0.091629,0,0.408409,0,0.147627,0,0.177628,0,0.250000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
231,CAN_231,COc1ccc(C[C@@](C)(N)C(=O)O)cc1OC,0.0,0,0.213106,0,0.151301,0,0.099695,0,0.288627,0,0.166667,0
232,CAN_232,CN1CCC(=C2c3ccccc3CCc3sccc32)CC1,0.0,0,0.072707,0,0.032488,0,0.106286,0,0.113880,0,0.083333,0
233,CAN_233,O=C(O)[C@]1(O)C[C@H](O)[C@@H](O)[C@@H](O)C1,0.0,1,0.707122,1,0.593930,1,0.824894,1,0.809364,1,0.750000,1
234,CAN_234,CCC(C)N1CCC2(CC1)N=C1C(=C3NC(=O)/C(C)=C/C=C/[C...,0.0,0,0.230783,0,0.342558,0,0.182650,0,0.060686,0,0.333333,0


In [27]:
df_nontoxic_top = df_nontoxic[(df_nontoxic["pred_rf_proba"] <=0.2) & (df_nontoxic["pred_lgbm_proba"]  <=0.2) & (df_nontoxic["pred_svm_proba"]  <=0.2)
& (df_nontoxic["pred_ridge_proba"]  <=0.2) & (df_nontoxic["pred_ridge_proba"]  <=0.2) ]
df_nontoxic_top

,Name,SMILES,Toxicity,pred_rf,pred_rf_proba,pred_lgbm,pred_lgbm_proba,pred_svm,pred_svm_proba,pred_ridge,pred_ridge_proba,pred_knn,pred_knn_proba,mode
56,CAN_56,CCCC(=O)Nc1ncnc2c1ncn2[C@@H]1O[C@@H]2OP(=O)(O)...,0.0,0,0.078710,0,0.156450,0,0.147682,0,0.079895,0,0.250000,0
57,CAN_57,CCCCCN(CCCCC)C(=O)C(CCC(=O)O)NC(=O)c1ccc(Cl)c(...,0.0,0,0.052293,0,0.101904,0,0.109446,0,0.094550,0,0.250000,0
66,CAN_66,C/C(=C(\CCOP(=O)(O)O)SC(=O)c1ccccc1)N(C=O)Cc1c...,0.0,0,0.159582,0,0.016911,0,0.119743,0,0.028320,0,0.083333,0
68,CAN_68,C[C@H](CCC(=O)O)[C@H]1CC[C@H]2[C@@H]3CC[C@@H]4...,0.0,0,0.134825,0,0.166155,0,0.089219,0,0.151623,0,0.333333,0
88,CAN_88,COc1cc(C(=O)OCCCN2CCCN(CCCOC(=O)c3cc(OC)c(OC)c...,0.0,0,0.089597,0,0.044136,0,0.075724,0,0.060061,0,0.000000,0
102,CAN_102,CN(C)CCOC(c1ccc(Cl)cc1)c1ccccn1,0.0,0,0.066312,0,0.073974,0,0.081072,0,0.128318,0,0.166667,0
110,CAN_110,COc1ccc2c(c1O)-c1c(OC)c(OC)cc3c1[C@H](C2)N(C)CC3,0.0,0,0.177237,0,0.066312,0,0.155005,0,0.177534,0,0.250000,0
112,CAN_112,COc1ccc(CC2c3cc(OC)c(OC)cc3CCN2C)cc1OC,0.0,0,0.097763,0,0.035792,0,0.077650,0,0.122213,0,0.166667,0
114,CAN_114,CN1C(CC(O)c2ccccc2)CCCC1CC(O)c1ccccc1,0.0,0,0.127871,0,0.163920,0,0.124487,0,0.137413,0,0.083333,0
139,CAN_139,COC(=O)[C@H]1[C@H]2C[C@@H]3c4[nH]c5cc(OC)ccc5c...,0.0,0,0.145526,0,0.137176,0,0.149318,0,0.120856,0,0.166667,0


In [28]:
mols2grid.display(df_nontoxic_top,smiles_col="SMILES",subset=["img","Name","Toxicity", "pred_ridge_proba", ],selection=False, addStereoAnnotation=True, size=(300, 250))

MolGridWidget()

In [29]:
df_nontoxic_last = df_nontoxic[(df_nontoxic["pred_rf_proba"] >=0.85) & (df_nontoxic["pred_lgbm_proba"]  >=0.85) & (df_nontoxic["pred_svm_proba"] >=0.85) 
& (df_nontoxic["pred_ridge_proba"] >=0.85) & (df_nontoxic["pred_knn_proba"]  >=0.85)]
df_nontoxic_last

,Name,SMILES,Toxicity,pred_rf,pred_rf_proba,pred_lgbm,pred_lgbm_proba,pred_svm,pred_svm_proba,pred_ridge,pred_ridge_proba,pred_knn,pred_knn_proba,mode


In [30]:
mols2grid.display(df_nontoxic_last,smiles_col="SMILES",subset=["img","Name","Toxicity", "pred_ridge_proba", ],selection=False, addStereoAnnotation=True, size=(300, 250))

MolGridWidget()

In [31]:
mat_met


,Metric,rf,lgbm,svm,ridge,knn,Combined Models
0,TP,31.000000,38.000000,32.000000,35.000000,33.000000,33.000000
1,TN,125.000000,127.000000,134.000000,130.000000,149.000000,130.000000
2,FP,63.000000,61.000000,54.000000,58.000000,39.000000,58.000000
3,FN,17.000000,10.000000,16.000000,13.000000,15.000000,15.000000
4,Accuracy,0.661017,0.699153,0.703390,0.699153,0.771186,0.690678
5,Precision,0.329787,0.383838,0.372093,0.376344,0.458333,0.362637
6,Sensitivity,0.645833,0.791667,0.666667,0.729167,0.687500,0.687500
7,Specificity,0.664900,0.675500,0.712800,0.691500,0.792600,0.691500
8,F1 score,0.436620,0.517007,0.477612,0.496454,0.550000,0.474820
9,F1 score (weighted),0.692297,0.727735,0.728773,0.726710,0.786267,0.718551
